# LightGBM Codex - Quantity-Only Full-Data Experiment

Final target and submission:
- `quantity` = forecasted purchase quantity for each `(location, item_id)` in `2026-01`.
- Primary metric = `MAPE_quantity`.

This experimental notebook keeps the main Codex notebook untouched, removes the revenue model/output, keeps only quantity-focused diagnostics, and defaults to full-data mode. Historical revenue columns remain usable as input features because they can help quantity forecasting. Calendar/Tet features are optional and disabled by default because prior runs did not improve quantity MAPE enough to justify the extra feature/RAM cost.


## Design

Temporal split for the new dataset:
- Train tune samples: target months `2025-07` to `2025-10`.
- Validation: `2025-11`, used for early stopping and calibration.
- Internal test: `2025-12`, because the new dataset has complete December transactions.
- Final train: target months `2025-07` to `2025-12`.
- Blind prediction: `2026-01`, using only history through `2025-12-31`.

Leakage rule:
- For target month `t`, all purchase and event features use only dates before the first day of `t`.
- `event_full_2025.parquet` uses `event_date` as behavior time.

In [ ]:
'''import os
print("Current working directory:", os.getcwd())
from google.colab import drive
drive.mount('/content/drive')'''

In [ ]:
import os
import gc
import json
import pickle
import warnings
from pathlib import Path
from datetime import timedelta

N_THREADS = int(os.environ.get("NOTEBOOK_THREADS", min(4, os.cpu_count() or 1)))
os.environ.setdefault("POLARS_MAX_THREADS", str(N_THREADS))
os.environ.setdefault("OMP_NUM_THREADS", str(N_THREADS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(N_THREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(N_THREADS))
os.environ.setdefault("NUMEXPR_NUM_THREADS", str(N_THREADS))

import numpy as np
import pandas as pd
import polars as pl
import lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error

warnings.filterwarnings("ignore")
pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(30)

SEED = 42
np.random.seed(SEED)

ROOT = Path.cwd()
for _ in range(8):
    if (ROOT / "raw_data").is_dir() and (ROOT / "Additional Doc").is_dir():
        break
    ROOT = ROOT.parent

DATA_DIR = ROOT / "raw_data" # Path("/content/drive/MyDrive/ColabData/lzu") 
ITEMS_PATH = DATA_DIR / "items.parquet"
TXN_PATH = DATA_DIR / "transaction_full_2025.parquet"
EVENT_PATH = DATA_DIR / "event_full_2025.parquet"
OUT = DATA_DIR / "Out-latet-Codex" # ROOT / "Additional Doc" / "Đồ-Án" / "Out-latet-Codex" 
OUT.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR = OUT / "_artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
RESUME_ARTIFACTS = os.environ.get("RESUME_ARTIFACTS", "1") != "0"
FORCE_RETRAIN = os.environ.get("FORCE_RETRAIN", "0") != "0"

TUNE_MODEL_PATH = ARTIFACT_DIR / "model_qty_tune.lgb"
TUNE_META_PATH = ARTIFACT_DIR / "tune_meta.json"
TUNE_RESULTS_PATH = ARTIFACT_DIR / "qty_weight_results.csv"
CALIB_META_PATH = ARTIFACT_DIR / "calibration_postprocess_meta.json"
POSTPROCESS_RESULTS_PATH = ARTIFACT_DIR / "qty_postprocess_results.csv"
FINAL_MODEL_PATH = ARTIFACT_DIR / "model_qty_final.lgb"
FINAL_META_PATH = ARTIFACT_DIR / "final_meta.json"
CAT_MAPS_FINAL_PATH = ARTIFACT_DIR / "cat_maps_final.pkl"
BLIND_IDS_PATH = ARTIFACT_DIR / "blind_ids.pkl"
BLIND_PRED_PATH = ARTIFACT_DIR / "pred_blind_qty.npy"

EXPERIMENT_VERSION = os.environ.get("EXPERIMENT_VERSION", "serious_lgb_param_ensemble_v2_sample_tune")
TUNE_CANDIDATE_DIR = ARTIFACT_DIR / "tune_candidates"
TUNE_CANDIDATE_DIR.mkdir(parents=True, exist_ok=True)
FINAL_ENSEMBLE_DIR = ARTIFACT_DIR / "final_ensemble_models"
FINAL_ENSEMBLE_DIR.mkdir(parents=True, exist_ok=True)
ENSEMBLE_META_PATH = ARTIFACT_DIR / "ensemble_meta.json"
BLIND_PRED_ENSEMBLE_PATH = ARTIFACT_DIR / "pred_blind_qty_ensemble.npy"


def _json_default(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    if isinstance(obj, Path):
        return str(obj)
    return str(obj)


def save_json(path, obj):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=_json_default), encoding="utf-8")
    tmp.replace(path)


def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

for p in [ITEMS_PATH, TXN_PATH, EVENT_PATH]:
    if not p.is_file():
        raise FileNotFoundError(p)

print("=" * 80)
print("LIGHTGBM LATEST CODEX - PATH CHECK")
print("=" * 80)
print(f"ROOT       : {ROOT}")
print(f"ITEMS      : {ITEMS_PATH}")
print(f"TXN        : {TXN_PATH}")
print(f"EVENT      : {EVENT_PATH}")
print(f"OUT        : {OUT}")
print(f"ARTIFACTS  : {ARTIFACT_DIR}")
print(f"Experiment : {EXPERIMENT_VERSION}")
print(f"Resume     : {RESUME_ARTIFACTS} / force retrain {FORCE_RETRAIN}")
print(f"Threads    : {N_THREADS}")
print("=" * 80)


In [ ]:
items_df = pl.read_parquet(str(ITEMS_PATH))
txn_lazy = pl.scan_parquet(str(TXN_PATH))
event_lazy = pl.scan_parquet(str(EVENT_PATH))

print("=" * 80)
print("SCHEMA")
print("=" * 80)
print("items")
print(items_df.schema)
print("\ntransactions")
print(txn_lazy.collect_schema())
print("\nevents")
print(event_lazy.collect_schema())

print("\nitems sale_status")
print(items_df.group_by("sale_status").len().sort("sale_status"))

print("\ntransaction event_type")
print(txn_lazy.group_by("event_type").len().sort("len", descending=True).collect())

print("\nevent event_type")
print(event_lazy.group_by("event_type").len().sort("len", descending=True).collect())

In [ ]:
def ym_expr(date_col):
    return (pl.col(date_col).dt.year() * 12 + pl.col(date_col).dt.month()).cast(pl.Int32)


def ym_to_label(ym):
    year = (ym - 1) // 12
    month = ((ym - 1) % 12) + 1
    return f"{year:04d}-{month:02d}"


YM_2025_01 = 2025 * 12 + 1
YM_2025_07 = 2025 * 12 + 7
YM_2025_10 = 2025 * 12 + 10
YM_2025_11 = 2025 * 12 + 11
YM_2025_12 = 2025 * 12 + 12
YM_2026_01 = 2026 * 12 + 1

TRAIN_TARGET_YMS = list(range(YM_2025_07, YM_2025_10 + 1))
VAL_YM = YM_2025_11
TEST_YM = YM_2025_12
FINAL_TARGET_YMS = list(range(YM_2025_07, YM_2025_12 + 1))
BLIND_YM = YM_2026_01

print("Train tune targets:", [ym_to_label(x) for x in TRAIN_TARGET_YMS])
print("Validation target :", ym_to_label(VAL_YM))
print("Internal test     :", ym_to_label(TEST_YM))
print("Final train       :", [ym_to_label(x) for x in FINAL_TARGET_YMS])
print("Blind target      :", ym_to_label(BLIND_YM))

## Preprocess

Transactions still compute historical revenue-related aggregates such as `rev_lag_*` and price/discount features because they can help the quantity model. No separate revenue target/model is trained in this experiment.


In [ ]:
items_active = (
    items_df
    .filter(pl.col("sale_status") != 0)
    .select([
        "item_id", "price", "category_l1", "category_l2", "category_l3",
        "category", "brand", "manufacturer", "size"
    ])
    .with_columns(pl.col("item_id").cast(pl.Utf8))
    .rename({"price": "item_list_price"})
)

active_item_ids = items_active.select("item_id").unique()

txn_clean = (
    txn_lazy
    .select([
        "bill_id", "customer_id", "item_id", "price", "location",
        "discount", "quantity", "event_type", "updated_date"
    ])
    .filter(pl.col("event_type").str.to_lowercase().str.contains("purchase"))
    .with_columns([
        pl.col("item_id").cast(pl.Utf8),
        pl.col("location").cast(pl.Int32),
        pl.col("updated_date").cast(pl.Datetime),
        pl.col("price").cast(pl.Float64).alias("price_f"),
        pl.col("discount").cast(pl.Float64).fill_null(0.0).alias("discount_f"),
        pl.col("quantity").cast(pl.Int64).alias("quantity_i"),
    ])
    .with_columns([
        pl.col("updated_date").dt.date().alias("date"),
        ym_expr("updated_date").alias("ym"),
        (pl.col("price_f") * pl.col("quantity_i").cast(pl.Float64)).alias("gross_revenue"),
    ])
    .with_columns([
        (pl.col("gross_revenue") - pl.col("discount_f")).clip(0, None).alias("revenue"),
        (pl.when(pl.col("gross_revenue") > 0)
          .then(pl.col("discount_f") / pl.col("gross_revenue"))
          .otherwise(0.0)).alias("discount_rate"),
    ])
    .filter(pl.col("ym").is_between(YM_2025_01, YM_2025_12))
    .join(active_item_ids.lazy(), on="item_id", how="inner")
)

txn_summary = txn_clean.select([
    pl.len().alias("rows"),
    pl.col("date").min().alias("min_date"),
    pl.col("date").max().alias("max_date"),
    pl.col("ym").n_unique().alias("n_months"),
    pl.col("location").n_unique().alias("n_locations"),
    pl.col("item_id").n_unique().alias("n_items"),
    pl.col("quantity_i").sum().alias("qty_sum"),
    pl.col("revenue").sum().alias("revenue_sum"),
]).collect()

print("=" * 80)
print("TRANSACTIONS AFTER FILTERING")
print("=" * 80)
print(txn_summary)

In [ ]:
event_clean = (
    event_lazy
    .select(["customer_id", "item_id", "price", "quantity", "event_type", "event_date"])
    .with_columns([
        pl.col("item_id").cast(pl.Utf8),
        pl.col("event_date").cast(pl.Datetime),
        pl.col("event_type").str.to_lowercase().alias("event_type_l"),
        pl.col("price").cast(pl.Float64).alias("event_price_f"),
        pl.col("quantity").cast(pl.Int64).alias("event_quantity_i"),
    ])
    .filter(pl.col("event_type_l").is_in(["view_item", "add_to_cart"]))
    .with_columns([
        pl.col("event_date").dt.date().alias("event_day"),
        ym_expr("event_date").alias("ym"),
    ])
    .filter(pl.col("ym").is_between(YM_2025_01, YM_2025_12))
    .join(active_item_ids.lazy(), on="item_id", how="inner")
)

event_summary = event_clean.select([
    pl.len().alias("rows"),
    pl.col("event_day").min().alias("min_event_day"),
    pl.col("event_day").max().alias("max_event_day"),
    pl.col("event_type_l").n_unique().alias("n_event_types"),
    pl.col("item_id").n_unique().alias("n_items"),
    pl.col("customer_id").n_unique().alias("n_customers"),
]).collect()

print("=" * 80)
print("EVENTS AFTER FILTERING")
print("=" * 80)
print(event_summary)
print(event_clean.group_by("event_type_l").len().sort("len", descending=True).collect())

## Aggregate Tables

The model forecasts monthly purchase quantity. It uses:
- monthly purchase history per `(location, item_id)`
- recent daily purchase windows per `(location, item_id)`
- monthly/recent event signals per `item_id` because event data has no location column
- item metadata and price/discount signals

In [ ]:
monthly = (
    txn_clean
    .group_by(["location", "item_id", "ym"])
    .agg([
        pl.col("quantity_i").sum().alias("qty"),
        pl.col("revenue").sum().alias("rev"),
        pl.col("gross_revenue").sum().alias("gross_rev"),
        pl.col("discount_f").sum().alias("discount_sum"),
        pl.col("discount_f").max().alias("discount_max"),
        pl.col("discount_rate").mean().alias("discount_rate_mean"),
        pl.col("date").n_unique().alias("active_days"),
        pl.col("bill_id").n_unique().alias("num_bills"),
        pl.col("customer_id").n_unique().alias("num_customers"),
        pl.col("price_f").mean().alias("avg_price"),
        pl.col("price_f").last().alias("last_price"),
    ])
    .with_columns([
        pl.col("qty").cast(pl.Float32),
        pl.col("rev").cast(pl.Float32),
        pl.col("gross_rev").cast(pl.Float32),
        pl.col("discount_sum").cast(pl.Float32),
        pl.col("discount_max").cast(pl.Float32),
        pl.col("discount_rate_mean").cast(pl.Float32),
        pl.col("active_days").cast(pl.Int16),
        pl.col("num_bills").cast(pl.Int32),
        pl.col("num_customers").cast(pl.Int32),
        pl.col("avg_price").cast(pl.Float32),
        pl.col("last_price").cast(pl.Float32),
    ])
    .collect()
)

purchase_daily = (
    txn_clean
    .group_by(["location", "item_id", "date"])
    .agg([
        pl.col("quantity_i").sum().alias("day_qty"),
        pl.col("revenue").sum().alias("day_rev"),
        pl.col("bill_id").n_unique().alias("day_transactions"),
    ])
    .with_columns([
        pl.col("day_qty").cast(pl.Float32),
        pl.col("day_rev").cast(pl.Float32),
        pl.col("day_transactions").cast(pl.Int32),
    ])
    .collect()
)

event_monthly = (
    event_clean
    .group_by(["item_id", "ym"])
    .agg([
        (pl.col("event_type_l") == "view_item").sum().alias("view_count"),
        (pl.col("event_type_l") == "add_to_cart").sum().alias("atc_count"),
        pl.when(pl.col("event_type_l") == "view_item").then(pl.col("event_quantity_i")).otherwise(0).sum().alias("view_qty"),
        pl.when(pl.col("event_type_l") == "add_to_cart").then(pl.col("event_quantity_i")).otherwise(0).sum().alias("atc_qty"),
        pl.col("customer_id").n_unique().alias("event_customers"),
        pl.col("event_price_f").mean().alias("event_avg_price"),
    ])
    .with_columns([
        pl.col("view_count").cast(pl.Int32),
        pl.col("atc_count").cast(pl.Int32),
        pl.col("view_qty").cast(pl.Float32),
        pl.col("atc_qty").cast(pl.Float32),
        pl.col("event_customers").cast(pl.Int32),
        pl.col("event_avg_price").cast(pl.Float32),
    ])
    .collect()
)

event_daily = (
    event_clean
    .group_by(["item_id", "event_day"])
    .agg([
        (pl.col("event_type_l") == "view_item").sum().alias("day_view_count"),
        (pl.col("event_type_l") == "add_to_cart").sum().alias("day_atc_count"),
        pl.col("customer_id").n_unique().alias("day_event_customers"),
    ])
    .with_columns([
        pl.col("day_view_count").cast(pl.Int32),
        pl.col("day_atc_count").cast(pl.Int32),
        pl.col("day_event_customers").cast(pl.Int32),
    ])
    .collect()
)


print("=" * 80)
print("AGGREGATES")
print("=" * 80)
print(f"monthly        : {monthly.shape}")
print(f"purchase_daily : {purchase_daily.shape}")
print(f"event_monthly  : {event_monthly.shape}")
print(f"event_daily    : {event_daily.shape}")
print(monthly.group_by("ym").agg(pl.col("qty").sum().alias("qty")).sort("ym").with_columns(
    pl.col("ym").map_elements(ym_to_label, return_dtype=pl.Utf8).alias("month")
).select(["month", "qty"]))

## Feature Engineering

The function below is leak-free. Every feature for a target month is built only from months/days before that target month.

In [ ]:
MAX_LAG = 12
ROLL_WINDOWS = [2, 3, 6, 9, 12]
RECENT_DAY_WINDOWS = [3, 7, 14, 30, 60, 90]


def _safe_div_expr(num_expr, den_expr):
    return pl.when(den_expr.abs() > 0).then(num_expr / den_expr).otherwise(0.0)


def _target_month_start(ym: int) -> pl.Date:
    year = (ym - 1) // 12
    month = ((ym - 1) % 12) + 1
    return pl.date(year, month, 1)


def _date_bounds_for_ym(ym: int):
    year = (ym - 1) // 12
    month = ((ym - 1) % 12) + 1
    return pd.Timestamp(year=year, month=month, day=1).date()


def _add_lag_features(ft, monthly_all, target_ym):
    out = ft
    for lag in range(1, MAX_LAG + 1):
        src_ym = target_ym - lag
        lag_df = (
            monthly_all
            .filter(pl.col("ym") == src_ym)
            .select([
                "location", "item_id",
                pl.col("qty").cast(pl.Float32).alias(f"qty_lag_{lag}m"),
                pl.col("rev").cast(pl.Float32).alias(f"rev_lag_{lag}m"),
                pl.col("active_days").cast(pl.Float32).alias(f"active_days_lag_{lag}m"),
                pl.col("num_bills").cast(pl.Float32).alias(f"bills_lag_{lag}m"),
                pl.col("num_customers").cast(pl.Float32).alias(f"customers_lag_{lag}m"),
                pl.col("discount_sum").cast(pl.Float32).alias(f"discount_lag_{lag}m"),
            ])
        )
        out = out.join(lag_df, on=["location", "item_id"], how="left")

    fill_cols = [c for c in out.columns if any(c.startswith(prefix) for prefix in [
        "qty_lag_", "rev_lag_", "active_days_lag_", "bills_lag_",
        "customers_lag_", "discount_lag_"
    ])]
    return out.with_columns([pl.col(c).fill_null(0.0).alias(c) for c in fill_cols])


def _add_event_lag_features(ft, event_monthly_all, target_ym):
    out = ft
    for lag in range(1, MAX_LAG + 1):
        src_ym = target_ym - lag
        ev = (
            event_monthly_all
            .filter(pl.col("ym") == src_ym)
            .select([
                "item_id",
                pl.col("view_count").cast(pl.Float32).alias(f"view_lag_{lag}m"),
                pl.col("atc_count").cast(pl.Float32).alias(f"atc_lag_{lag}m"),
                pl.col("view_qty").cast(pl.Float32).alias(f"view_qty_lag_{lag}m"),
                pl.col("atc_qty").cast(pl.Float32).alias(f"atc_qty_lag_{lag}m"),
                pl.col("event_customers").cast(pl.Float32).alias(f"event_customers_lag_{lag}m"),
            ])
        )
        out = out.join(ev, on="item_id", how="left")

    fill_cols = [c for c in out.columns if any(c.startswith(prefix) for prefix in [
        "view_lag_", "atc_lag_", "view_qty_lag_", "atc_qty_lag_", "event_customers_lag_"
    ])]
    return out.with_columns([pl.col(c).fill_null(0.0).alias(c) for c in fill_cols])


def _horizontal_std(cols):
    return pl.concat_list(cols).list.eval(pl.element().std()).list.first()


def _add_rolling_features(ft):
    out = ft
    q_lags = [f"qty_lag_{i}m" for i in range(1, MAX_LAG + 1)]
    r_lags = [f"rev_lag_{i}m" for i in range(1, MAX_LAG + 1)]
    view_lags = [f"view_lag_{i}m" for i in range(1, MAX_LAG + 1)]
    atc_lags = [f"atc_lag_{i}m" for i in range(1, MAX_LAG + 1)]
    bill_lags = [f"bills_lag_{i}m" for i in range(1, MAX_LAG + 1)]

    exprs = [
        pl.sum_horizontal(q_lags).alias("qty_total_hist_12m"),
        (pl.sum_horizontal(q_lags) / MAX_LAG).alias("qty_mean_hist_12m"),
        _horizontal_std(q_lags).alias("qty_std_hist_12m"),
        pl.max_horizontal(q_lags).alias("qty_max_hist_12m"),
        pl.sum_horizontal(r_lags).alias("rev_total_hist_12m"),
        (pl.sum_horizontal(r_lags) / MAX_LAG).alias("rev_mean_hist_12m"),
        _horizontal_std(r_lags).alias("rev_std_hist_12m"),
        pl.max_horizontal(r_lags).alias("rev_max_hist_12m"),
        pl.sum_horizontal([pl.col(c) > 0 for c in q_lags]).alias("months_active_12m"),
        pl.sum_horizontal([pl.col(f"qty_lag_{i}m") > 0 for i in range(1, 4)]).alias("months_active_3m"),
        pl.sum_horizontal([pl.col(f"qty_lag_{i}m") > 0 for i in range(1, 7)]).alias("months_active_6m"),
        pl.sum_horizontal([pl.col(f"qty_lag_{i}m") for i in range(1, 4)]).alias("qty_recent_3m"),
        pl.sum_horizontal([pl.col(f"rev_lag_{i}m") for i in range(1, 4)]).alias("rev_recent_3m"),
        pl.sum_horizontal(bill_lags).alias("n_transactions_12m"),
        pl.sum_horizontal(view_lags).alias("view_total_12m"),
        pl.sum_horizontal(atc_lags).alias("atc_total_12m"),
    ]

    for w in ROLL_WINDOWS:
        q = q_lags[:w]
        r = r_lags[:w]
        v = view_lags[:w]
        a = atc_lags[:w]
        exprs.extend([
            (pl.sum_horizontal(q) / w).alias(f"qty_roll_mean_{w}m"),
            _horizontal_std(q).alias(f"qty_roll_std_{w}m"),
            pl.max_horizontal(q).alias(f"qty_roll_max_{w}m"),
            (pl.sum_horizontal(r) / w).alias(f"rev_roll_mean_{w}m"),
            _horizontal_std(r).alias(f"rev_roll_std_{w}m"),
            pl.max_horizontal(r).alias(f"rev_roll_max_{w}m"),
            pl.sum_horizontal(v).alias(f"view_roll_sum_{w}m"),
            pl.sum_horizontal(a).alias(f"atc_roll_sum_{w}m"),
            _safe_div_expr(pl.sum_horizontal(a), pl.sum_horizontal(v)).alias(f"atc_view_ratio_{w}m"),
        ])

    out = out.with_columns(exprs)

    recency = pl.lit(MAX_LAG + 1)
    for lag in range(MAX_LAG, 0, -1):
        recency = pl.when(pl.col(f"qty_lag_{lag}m") > 0).then(pl.lit(lag)).otherwise(recency)

    return out.with_columns([
        recency.alias("months_since_last_sale"),
        (MAX_LAG - pl.col("months_active_12m")).alias("zero_months_12m"),
        (pl.lit(3) - pl.col("months_active_3m")).alias("zero_months_3m"),
        (pl.lit(6) - pl.col("months_active_6m")).alias("zero_months_6m"),
        _safe_div_expr(pl.col("qty_std_hist_12m"), pl.col("qty_mean_hist_12m")).alias("qty_cv_12m"),
        _safe_div_expr(pl.col("rev_std_hist_12m"), pl.col("rev_mean_hist_12m")).alias("rev_cv_12m"),
        (pl.col("qty_lag_1m") - pl.col("qty_lag_2m")).alias("trend_qty_1m_2m"),
        (pl.col("qty_roll_mean_3m") - pl.col("qty_roll_mean_6m")).alias("trend_qty_3m_6m"),
        _safe_div_expr(pl.col("qty_roll_mean_3m"), pl.col("qty_roll_mean_6m")).alias("qty_roll_3m_6m_ratio"),
        (pl.col("qty_lag_1m") - 2 * pl.col("qty_lag_2m") + pl.col("qty_lag_3m")).alias("accel_qty_3m"),
        (pl.col("rev_lag_1m") - pl.col("rev_lag_2m")).alias("trend_rev_1m_2m"),
        (pl.col("rev_roll_mean_3m") - pl.col("rev_roll_mean_6m")).alias("trend_rev_3m_6m"),
        _safe_div_expr(pl.col("rev_roll_mean_3m"), pl.col("rev_roll_mean_6m")).alias("rev_roll_3m_6m_ratio"),
    ])


def _add_recent_daily_features(ft, target_ym):
    target_start = _date_bounds_for_ym(target_ym)
    out = ft
    for days in RECENT_DAY_WINDOWS:
        start = target_start - pd.Timedelta(days=days)
        p_recent = (
            purchase_daily
            .filter((pl.col("date") >= start) & (pl.col("date") < target_start))
            .group_by(["location", "item_id"])
            .agg([
                pl.col("day_qty").sum().cast(pl.Float32).alias(f"qty_last_{days}d"),
                pl.col("day_rev").sum().cast(pl.Float32).alias(f"rev_last_{days}d"),
                pl.col("day_transactions").sum().cast(pl.Float32).alias(f"n_transactions_last_{days}d"),
                (pl.col("day_qty") > 0).sum().cast(pl.Float32).alias(f"sale_active_days_last_{days}d"),
                pl.col("day_qty").mean().cast(pl.Float32).alias(f"qty_daily_mean_{days}d"),
                pl.col("day_qty").std().fill_null(0).cast(pl.Float32).alias(f"qty_daily_std_{days}d"),
                pl.col("day_qty").max().cast(pl.Float32).alias(f"qty_daily_max_{days}d"),
                pl.col("day_rev").max().cast(pl.Float32).alias(f"rev_daily_max_{days}d"),
            ])
        )
        out = out.join(p_recent, on=["location", "item_id"], how="left")

        e_recent = (
            event_daily
            .filter((pl.col("event_day") >= start) & (pl.col("event_day") < target_start))
            .group_by("item_id")
            .agg([
                pl.col("day_view_count").sum().cast(pl.Float32).alias(f"view_last_{days}d"),
                pl.col("day_atc_count").sum().cast(pl.Float32).alias(f"atc_last_{days}d"),
                pl.col("day_event_customers").sum().cast(pl.Float32).alias(f"event_customers_last_{days}d"),
                (pl.col("day_view_count") > 0).sum().cast(pl.Float32).alias(f"view_active_days_last_{days}d"),
                (pl.col("day_atc_count") > 0).sum().cast(pl.Float32).alias(f"atc_active_days_last_{days}d"),
                pl.col("day_view_count").max().cast(pl.Float32).alias(f"view_daily_max_{days}d"),
                pl.col("day_atc_count").max().cast(pl.Float32).alias(f"atc_daily_max_{days}d"),
            ])
        )
        out = out.join(e_recent, on="item_id", how="left")

    recent_cols = [c for c in out.columns if any(tag in c for tag in ["_last_", "_daily_mean_", "_daily_std_"])]
    out = out.with_columns([pl.col(c).fill_null(0.0).alias(c) for c in recent_cols])

    ratio_exprs = []
    for days in RECENT_DAY_WINDOWS:
        ratio_exprs.extend([
            _safe_div_expr(pl.col(f"atc_last_{days}d"), pl.col(f"view_last_{days}d")).alias(f"atc_view_ratio_{days}d"),
            _safe_div_expr(pl.col(f"qty_last_{days}d"), pl.col(f"view_last_{days}d")).alias(f"purchase_view_ratio_{days}d"),
            _safe_div_expr(pl.col(f"qty_last_{days}d"), pl.col(f"atc_last_{days}d")).alias(f"purchase_atc_ratio_{days}d"),
            (pl.col(f"atc_last_{days}d") + 0.03 * pl.col(f"view_last_{days}d")).alias(f"event_pressure_last_{days}d"),
        ])

    if all(c in out.columns for c in ["qty_last_7d", "qty_last_30d", "qty_last_14d", "qty_last_60d"]):
        ratio_exprs.extend([
            _safe_div_expr(pl.col("qty_last_7d"), pl.col("qty_last_30d")).alias("qty_last_7d_share_30d"),
            _safe_div_expr(pl.col("rev_last_7d"), pl.col("rev_last_30d")).alias("rev_last_7d_share_30d"),
            _safe_div_expr(pl.col("view_last_7d"), pl.col("view_last_30d")).alias("view_last_7d_share_30d"),
            _safe_div_expr(pl.col("atc_last_7d"), pl.col("atc_last_30d")).alias("atc_last_7d_share_30d"),
            _safe_div_expr(pl.col("qty_last_14d"), pl.col("qty_last_60d")).alias("qty_last_14d_share_60d"),
            _safe_div_expr(pl.col("rev_last_14d"), pl.col("rev_last_60d")).alias("rev_last_14d_share_60d"),
            _safe_div_expr(pl.col("view_last_14d"), pl.col("view_last_60d")).alias("view_last_14d_share_60d"),
            _safe_div_expr(pl.col("atc_last_14d"), pl.col("atc_last_60d")).alias("atc_last_14d_share_60d"),
            (pl.col("atc_last_7d") - pl.col("atc_last_30d") * (7.0 / 30.0)).alias("atc_7d_vs_30d_expected"),
            (pl.col("view_last_7d") - pl.col("view_last_30d") * (7.0 / 30.0)).alias("view_7d_vs_30d_expected"),
            (pl.col("qty_last_7d") - pl.col("qty_last_30d") * (7.0 / 30.0)).alias("qty_7d_vs_30d_expected"),
        ])
    return out.with_columns(ratio_exprs)


def _add_recency_features(ft, target_ym):
    target_start = _date_bounds_for_ym(target_ym)
    out = ft

    last_purchase = (
        purchase_daily
        .filter((pl.col("date") < target_start) & (pl.col("day_qty") > 0))
        .group_by(["location", "item_id"])
        .agg(pl.col("date").max().alias("last_purchase_day"))
    )
    out = out.join(last_purchase, on=["location", "item_id"], how="left")
    out = out.with_columns(
        pl.when(pl.col("last_purchase_day").is_not_null())
        .then((pl.lit(target_start) - pl.col("last_purchase_day")).dt.total_days())
        .otherwise(999)
        .cast(pl.Float32)
        .alias("days_since_last_purchase")
    ).drop("last_purchase_day")

    for count_col, alias in [
        ("day_view_count", "days_since_last_view"),
        ("day_atc_count", "days_since_last_atc"),
    ]:
        last_event = (
            event_daily
            .filter((pl.col("event_day") < target_start) & (pl.col(count_col) > 0))
            .group_by("item_id")
            .agg(pl.col("event_day").max().alias("last_event_day"))
        )
        out = out.join(last_event, on="item_id", how="left")
        out = out.with_columns(
            pl.when(pl.col("last_event_day").is_not_null())
            .then((pl.lit(target_start) - pl.col("last_event_day")).dt.total_days())
            .otherwise(999)
            .cast(pl.Float32)
            .alias(alias)
        ).drop("last_event_day")

    return out.with_columns([
        pl.min_horizontal("days_since_last_view", "days_since_last_atc").alias("days_since_last_event_any"),
        _safe_div_expr(pl.col("days_since_last_atc"), pl.col("days_since_last_view") + 1).alias("atc_view_recency_ratio"),
    ])


In [ ]:
ENABLE_HOLIDAY_FEATURES = os.environ.get("ENABLE_HOLIDAY_FEATURES", "0") != "0"

TET_DATES = {
    2025: pd.Timestamp("2025-01-29").date(),
    2026: pd.Timestamp("2026-02-17").date(),
}


def _month_bounds_from_ym(ym: int):
    year = (ym - 1) // 12
    month = ((ym - 1) % 12) + 1
    start = pd.Timestamp(year=year, month=month, day=1).date()
    end = (pd.Timestamp(year=year, month=month, day=1) + pd.offsets.MonthEnd(0)).date()
    return start, end


def _overlap_days(a_start, a_end, b_start, b_end) -> int:
    start = max(a_start, b_start)
    end = min(a_end, b_end)
    if end < start:
        return 0
    return (end - start).days + 1


def _black_friday(year: int):
    fridays = [d.date() for d in pd.date_range(f"{year}-11-01", f"{year}-11-30", freq="D") if d.weekday() == 4]
    return fridays[3]


def _calendar_features_for_ym(ym: int) -> dict[str, float]:
    month_start, month_end = _month_bounds_from_ym(ym)
    year = month_start.year

    promo_dates = [
        pd.Timestamp(year=year, month=1, day=1).date(),
        pd.Timestamp(year=year, month=10, day=20).date(),
        pd.Timestamp(year=year, month=11, day=11).date(),
        _black_friday(year),
        pd.Timestamp(year=year, month=12, day=12).date(),
        pd.Timestamp(year=year, month=12, day=24).date(),
        pd.Timestamp(year=year, month=12, day=25).date(),
    ]
    official_dates = [
        pd.Timestamp(year=year, month=1, day=1).date(),
        pd.Timestamp(year=year, month=4, day=30).date(),
        pd.Timestamp(year=year, month=5, day=1).date(),
        pd.Timestamp(year=year, month=9, day=1).date(),
        pd.Timestamp(year=year, month=9, day=2).date(),
    ]
    big_sale_dates = [
        pd.Timestamp(year=year, month=11, day=11).date(),
        _black_friday(year),
        pd.Timestamp(year=year, month=12, day=12).date(),
    ]

    next_tets = [d for d in TET_DATES.values() if d >= month_start]
    next_tet = min(next_tets) if next_tets else max(TET_DATES.values())

    feats = {
        "promo_days_in_month": sum(month_start <= d <= month_end for d in promo_dates),
        "big_sale_days_in_month": sum(month_start <= d <= month_end for d in big_sale_dates),
        "official_holiday_days_in_month": sum(month_start <= d <= month_end for d in official_dates),
        "is_big_sale_month": int(any(month_start <= d <= month_end for d in big_sale_dates)),
        "days_to_next_tet_from_month_start": (next_tet - month_start).days,
        "days_to_next_tet_from_month_end": (next_tet - month_end).days,
    }

    for window in [30, 45, 60, 90]:
        pre_start = next_tet - timedelta(days=window)
        pre_end = next_tet - timedelta(days=1)
        overlap = _overlap_days(month_start, month_end, pre_start, pre_end)
        feats[f"pre_tet_{window}d_overlap"] = overlap
        feats[f"is_pre_tet_{window}d_month"] = int(overlap > 0)

    tet_overlap = _overlap_days(month_start, month_end, next_tet, next_tet + timedelta(days=6))
    post_tet_overlap = _overlap_days(month_start, month_end, next_tet + timedelta(days=1), next_tet + timedelta(days=30))
    feats["tet_7d_overlap"] = tet_overlap
    feats["post_tet_30d_overlap"] = post_tet_overlap
    feats["is_tet_month"] = int(tet_overlap > 0)
    feats["is_post_tet_month"] = int(post_tet_overlap > 0)
    return feats


def optimize_feature_dtypes(df: pl.DataFrame) -> pl.DataFrame:
    exprs = []
    for name, dtype in df.schema.items():
        if dtype == pl.Float64:
            exprs.append(pl.col(name).cast(pl.Float32).alias(name))
        elif dtype == pl.Int64:
            exprs.append(pl.col(name).cast(pl.Int32).alias(name))
        elif dtype == pl.UInt64:
            exprs.append(pl.col(name).cast(pl.UInt32).alias(name))
    return df.with_columns(exprs) if exprs else df


def _sample_skeleton_before_features(skeleton: pl.DataFrame, labels: pl.DataFrame | None, max_rows: int, seed_offset: int = 0) -> pl.DataFrame:
    """Apply positive/zero sampling before expensive feature joins."""
    if max_rows <= 0 or skeleton.height <= max_rows:
        return skeleton

    if labels is None:
        return skeleton.sample(n=max_rows, seed=SEED + seed_offset)

    labeled = (
        skeleton
        .join(labels.select(["location", "item_id", "y_qty"]), on=["location", "item_id"], how="left")
        .with_columns(pl.col("y_qty").fill_null(0.0))
    )
    pos = labeled.filter(pl.col("y_qty") > 0)
    zero = labeled.filter(pl.col("y_qty") <= 0)
    pos_target = min(pos.height, int(max_rows * 0.65))
    zero_target = max_rows - pos_target
    parts = []
    if pos_target > 0:
        parts.append(pos.sample(n=pos_target, seed=SEED + seed_offset) if pos.height > pos_target else pos)
    if zero_target > 0 and zero.height > 0:
        keep = min(zero_target, zero.height)
        parts.append(zero.sample(n=keep, seed=SEED + 100 + seed_offset) if zero.height > keep else zero)

    sampled = pl.concat(parts, how="vertical_relaxed") if parts else labeled.head(0)
    if sampled.height < max_rows:
        remaining = labeled.join(sampled.select(["location", "item_id"]), on=["location", "item_id"], how="anti")
        extra_n = min(max_rows - sampled.height, remaining.height)
        if extra_n > 0:
            sampled = pl.concat([
                sampled,
                remaining.sample(n=extra_n, seed=SEED + 300 + seed_offset) if remaining.height > extra_n else remaining,
            ], how="vertical_relaxed")

    return (
        sampled
        .sample(fraction=1.0, shuffle=True, seed=SEED + 200 + seed_offset)
        .select(["location", "item_id", "item_id_code"])
    )


def build_features(
    target_ym: int,
    include_target: bool = True,
    skeleton_max_rows: int = 0,
    seed_offset: int = 0,
    skeleton_keys: pl.DataFrame | None = None,
):
    history = monthly.filter(pl.col("ym") < target_ym)
    target = monthly.filter(pl.col("ym") == target_ym)

    labels = None
    if include_target:
        labels = target.group_by(["location", "item_id"]).agg([
            pl.col("qty").sum().cast(pl.Float32).alias("y_qty"),
        ])

    if skeleton_keys is None:
        skeleton = history.select(["location", "item_id"]).unique()
    else:
        skeleton = skeleton_keys.select(["location", "item_id"]).unique()

    skeleton = skeleton.with_columns([
        pl.col("location").cast(pl.Int32),
        pl.col("item_id").alias("item_id_code"),
    ])
    raw_skeleton_rows = skeleton.height
    skeleton = _sample_skeleton_before_features(skeleton, labels, skeleton_max_rows, seed_offset=seed_offset)
    if skeleton.height != raw_skeleton_rows:
        print(f"  skeleton: {raw_skeleton_rows:,} -> {skeleton.height:,} before feature joins")

    ft = _add_lag_features(skeleton, monthly, target_ym)
    ft = _add_event_lag_features(ft, event_monthly, target_ym)
    ft = _add_rolling_features(ft)
    ft = _add_recent_daily_features(ft, target_ym)
    ft = _add_recency_features(ft, target_ym)

    hist_agg = (
        history
        .group_by(["location", "item_id"])
        .agg([
            pl.col("active_days").sum().cast(pl.Float32).alias("active_days_hist"),
            pl.col("num_bills").sum().cast(pl.Float32).alias("num_bills_hist"),
            pl.col("num_customers").sum().cast(pl.Float32).alias("num_customers_hist"),
            pl.col("avg_price").mean().cast(pl.Float32).alias("avg_price_hist"),
            pl.col("last_price").last().cast(pl.Float32).alias("last_known_price"),
            pl.col("discount_sum").sum().cast(pl.Float32).alias("discount_sum_hist"),
            pl.col("discount_max").max().cast(pl.Float32).alias("discount_max_hist"),
            pl.col("discount_rate_mean").mean().cast(pl.Float32).alias("discount_rate_mean_hist"),
        ])
    )
    ft = ft.join(hist_agg, on=["location", "item_id"], how="left")

    ft = ft.join(items_active, on="item_id", how="left")
    ft = ft.with_columns([
        pl.coalesce("last_known_price", "avg_price_hist", "item_list_price").cast(pl.Float32).alias("price_ref")
    ])

    cat_price = (
        items_active
        .group_by("category")
        .agg(pl.col("item_list_price").cast(pl.Float32).median().alias("category_median_price"))
    )
    ft = ft.join(cat_price, on="category", how="left")

    recent_3m = history.filter(pl.col("ym") >= target_ym - 3)
    recent_6m = history.filter(pl.col("ym") >= target_ym - 6)

    item_global = history.group_by("item_id").agg([
        pl.col("qty").sum().cast(pl.Float32).alias("item_qty_hist"),
        pl.col("rev").sum().cast(pl.Float32).alias("item_rev_hist"),
    ])
    item_recent = recent_3m.group_by("item_id").agg([
        pl.col("qty").sum().cast(pl.Float32).alias("item_qty_recent_3m"),
        pl.col("rev").sum().cast(pl.Float32).alias("item_rev_recent_3m"),
    ])
    item_recent_6m = recent_6m.group_by("item_id").agg([
        pl.col("qty").sum().cast(pl.Float32).alias("item_qty_recent_6m"),
        pl.col("rev").sum().cast(pl.Float32).alias("item_rev_recent_6m"),
    ])
    loc_global = history.group_by("location").agg([
        pl.col("qty").sum().cast(pl.Float32).alias("loc_qty_hist"),
        pl.col("rev").sum().cast(pl.Float32).alias("loc_rev_hist"),
    ])
    loc_recent = recent_3m.group_by("location").agg([
        pl.col("qty").sum().cast(pl.Float32).alias("loc_qty_recent_3m"),
        pl.col("rev").sum().cast(pl.Float32).alias("loc_rev_recent_3m"),
    ])
    loc_recent_6m = recent_6m.group_by("location").agg([
        pl.col("qty").sum().cast(pl.Float32).alias("loc_qty_recent_6m"),
        pl.col("rev").sum().cast(pl.Float32).alias("loc_rev_recent_6m"),
    ])
    hist_meta = history.join(items_active.select(["item_id", "category", "brand"]), on="item_id", how="left")
    recent_meta = recent_3m.join(items_active.select(["item_id", "category", "brand"]), on="item_id", how="left")
    cat_global = hist_meta.group_by("category").agg([
        pl.col("qty").sum().cast(pl.Float32).alias("category_qty_hist"),
        pl.col("rev").sum().cast(pl.Float32).alias("category_rev_hist"),
    ])
    cat_recent = recent_meta.group_by("category").agg([
        pl.col("qty").sum().cast(pl.Float32).alias("category_qty_recent_3m"),
        pl.col("rev").sum().cast(pl.Float32).alias("category_rev_recent_3m"),
    ])
    loc_cat = hist_meta.group_by(["location", "category"]).agg([
        pl.col("qty").sum().cast(pl.Float32).alias("loc_category_qty_hist"),
        pl.col("rev").sum().cast(pl.Float32).alias("loc_category_rev_hist"),
    ])
    loc_cat_recent = recent_meta.group_by(["location", "category"]).agg([
        pl.col("qty").sum().cast(pl.Float32).alias("loc_category_qty_recent_3m"),
        pl.col("rev").sum().cast(pl.Float32).alias("loc_category_rev_recent_3m"),
    ])
    brand_global = hist_meta.group_by("brand").agg([
        pl.col("qty").sum().cast(pl.Float32).alias("brand_qty_hist"),
        pl.col("rev").sum().cast(pl.Float32).alias("brand_rev_hist"),
    ])
    brand_recent = recent_meta.group_by("brand").agg([
        pl.col("qty").sum().cast(pl.Float32).alias("brand_qty_recent_3m"),
        pl.col("rev").sum().cast(pl.Float32).alias("brand_rev_recent_3m"),
    ])

    ft = (
        ft.join(item_global, on="item_id", how="left")
          .join(item_recent, on="item_id", how="left")
          .join(item_recent_6m, on="item_id", how="left")
          .join(loc_global, on="location", how="left")
          .join(loc_recent, on="location", how="left")
          .join(loc_recent_6m, on="location", how="left")
          .join(cat_global, on="category", how="left")
          .join(cat_recent, on="category", how="left")
          .join(loc_cat, on=["location", "category"], how="left")
          .join(loc_cat_recent, on=["location", "category"], how="left")
          .join(brand_global, on="brand", how="left")
          .join(brand_recent, on="brand", how="left")
          .fill_null(0)
    )

    target_year = (target_ym - 1) // 12
    target_month = ((target_ym - 1) % 12) + 1
    calendar_feature_values = _calendar_features_for_ym(target_ym) if ENABLE_HOLIDAY_FEATURES else {}
    calendar_exprs = [
        pl.lit(v).cast(pl.Float32).alias(k)
        for k, v in calendar_feature_values.items()
    ]
    ft = ft.with_columns([
        pl.lit(target_ym).cast(pl.Int32).alias("target_ym"),
        pl.lit(target_year).cast(pl.Int32).alias("target_year"),
        pl.lit(target_month).cast(pl.Int32).alias("target_month"),
        pl.lit(len([m for m in range(target_ym - MAX_LAG, target_ym) if m >= YM_2025_01])).cast(pl.Int32).alias("history_months_count"),
        *calendar_exprs,
        pl.col("price_ref").log1p().alias("log_price_ref"),
        pl.col("avg_price_hist").log1p().alias("log_avg_price_hist"),
        pl.col("item_list_price").cast(pl.Float32).log1p().alias("log_catalog_price"),
        _safe_div_expr(pl.col("price_ref"), pl.col("category_median_price")).alias("price_vs_category_median"),
        _safe_div_expr(pl.col("rev_total_hist_12m"), pl.col("qty_total_hist_12m")).alias("unit_revenue_hist"),
        _safe_div_expr(pl.col("rev_recent_3m"), pl.col("qty_recent_3m")).alias("unit_revenue_recent_3m"),
        _safe_div_expr(pl.col("discount_sum_hist"), pl.col("rev_total_hist_12m").abs() + 1).alias("discount_intensity_hist"),
        _safe_div_expr(pl.col("discount_max_hist"), pl.col("price_ref").abs() + 1).alias("max_discount_vs_price"),
        _safe_div_expr(pl.col("qty_total_hist_12m"), pl.col("item_qty_hist")).alias("loc_item_qty_share_of_item"),
        _safe_div_expr(pl.col("qty_total_hist_12m"), pl.col("loc_qty_hist")).alias("loc_item_qty_share_of_location"),
        _safe_div_expr(pl.col("rev_total_hist_12m"), pl.col("item_rev_hist")).alias("loc_item_rev_share_of_item"),
        _safe_div_expr(pl.col("rev_total_hist_12m"), pl.col("loc_rev_hist")).alias("loc_item_rev_share_of_location"),
        _safe_div_expr(pl.col("loc_category_qty_hist"), pl.col("category_qty_hist")).alias("loc_category_qty_share"),
        _safe_div_expr(pl.col("loc_category_rev_hist"), pl.col("category_rev_hist")).alias("loc_category_rev_share"),
        _safe_div_expr(pl.col("qty_recent_3m"), pl.col("item_qty_recent_3m")).alias("loc_item_qty_share_recent_3m"),
        _safe_div_expr(pl.col("rev_recent_3m"), pl.col("item_rev_recent_3m")).alias("loc_item_rev_share_recent_3m"),
        _safe_div_expr(pl.col("qty_recent_3m"), pl.col("loc_qty_recent_3m")).alias("loc_item_qty_share_of_location_recent_3m"),
        _safe_div_expr(pl.col("loc_category_qty_recent_3m"), pl.col("category_qty_recent_3m")).alias("loc_category_qty_share_recent_3m"),
        _safe_div_expr(pl.col("brand_qty_recent_3m"), pl.col("item_qty_recent_3m")).alias("brand_item_recent_ratio"),
    ])

    if ENABLE_HOLIDAY_FEATURES and "pre_tet_60d_overlap" in ft.columns:
        ft = ft.with_columns([
            (pl.col("pre_tet_60d_overlap") * pl.col("qty_last_30d")).alias("pre_tet_60d_x_qty_last_30d"),
            (pl.col("pre_tet_60d_overlap") * pl.col("rev_last_30d")).alias("pre_tet_60d_x_rev_last_30d"),
            (pl.col("pre_tet_60d_overlap") * pl.col("view_last_30d")).alias("pre_tet_60d_x_view_last_30d"),
            (pl.col("pre_tet_60d_overlap") * pl.col("atc_last_30d")).alias("pre_tet_60d_x_atc_last_30d"),
            (pl.col("big_sale_days_in_month") * pl.col("view_last_14d")).alias("big_sale_x_view_last_14d"),
            (pl.col("big_sale_days_in_month") * pl.col("atc_last_14d")).alias("big_sale_x_atc_last_14d"),
        ])

    if include_target:
        ft = ft.join(labels, on=["location", "item_id"], how="left")
        ft = ft.with_columns([
            pl.col("y_qty").fill_null(0.0),
        ])
    else:
        ft = ft.with_columns([
            pl.lit(0.0).alias("y_qty"),
        ])

    return optimize_feature_dtypes(ft)


sample_ft = build_features(YM_2025_07, include_target=True, skeleton_max_rows=50_000, seed_offset=YM_2025_07)
print("=" * 80)
print("FEATURE FUNCTION CHECK")
print("=" * 80)
print(sample_ft.shape)
print(sample_ft.select(["location", "item_id", "target_ym", "y_qty"]).head(5))
del sample_ft
gc.collect()

## Build Train / Validation / Deferred Test

Use environment variables to cap rows during development:
- `MAX_TUNE_ROWS`
- `MAX_VAL_ROWS`
- `MAX_TEST_ROWS`
- `MAX_FINAL_ROWS`

This notebook defaults to RAM-safe caps. Set a cap to `0` only on a high-RAM machine to run the full split. Sampling is applied before expensive feature joins, so the cap actually reduces peak RAM.

In [ ]:
LOW_MEMORY_MODE = os.environ.get("LOW_MEMORY_MODE", "0") != "0"
DEFAULT_TUNE_ROWS = 800_000 if LOW_MEMORY_MODE else 1_500_000
DEFAULT_VAL_ROWS = 250_000 if LOW_MEMORY_MODE else 0
DEFAULT_TEST_ROWS = 250_000 if LOW_MEMORY_MODE else 0
DEFAULT_FINAL_ROWS = 1_200_000 if LOW_MEMORY_MODE else 0

MAX_TUNE_ROWS = int(os.environ.get("MAX_TUNE_ROWS", str(DEFAULT_TUNE_ROWS)))
MAX_VAL_ROWS = int(os.environ.get("MAX_VAL_ROWS", str(DEFAULT_VAL_ROWS)))
MAX_TEST_ROWS = int(os.environ.get("MAX_TEST_ROWS", str(DEFAULT_TEST_ROWS)))
MAX_FINAL_ROWS = int(os.environ.get("MAX_FINAL_ROWS", str(DEFAULT_FINAL_ROWS)))
FEATURE_CACHE_DIR = OUT / "_feature_cache"
FEATURE_CACHE_DIR.mkdir(parents=True, exist_ok=True)


def sample_supervised(df: pl.DataFrame, max_rows: int, seed_offset: int = 0) -> pl.DataFrame:
    if max_rows <= 0 or df.height <= max_rows:
        return df

    row_id = "__sample_row_id"
    work = df.with_row_index(row_id)
    pos = work.filter(pl.col("y_qty") > 0)
    zero = work.filter(pl.col("y_qty") <= 0)
    pos_target = min(pos.height, int(max_rows * 0.65))
    zero_target = max_rows - pos_target
    parts = []
    if pos_target > 0:
        parts.append(pos.sample(n=pos_target, seed=SEED + seed_offset) if pos.height > pos_target else pos)
    if zero_target > 0 and zero.height > 0:
        keep = min(zero_target, zero.height)
        parts.append(zero.sample(n=keep, seed=SEED + 100 + seed_offset) if zero.height > keep else zero)

    sampled = pl.concat(parts, how="vertical_relaxed") if parts else work.head(0)
    if sampled.height < max_rows:
        remaining = work.join(sampled.select(row_id), on=row_id, how="anti")
        extra_n = min(max_rows - sampled.height, remaining.height)
        if extra_n > 0:
            sampled = pl.concat([
                sampled,
                remaining.sample(n=extra_n, seed=SEED + 300 + seed_offset) if remaining.height > extra_n else remaining,
            ], how="vertical_relaxed")

    return (
        sampled
        .sample(fraction=1.0, shuffle=True, seed=SEED + 200 + seed_offset)
        .drop(row_id)
    )


def _feature_cache_path(prefix: str, ym: int) -> Path:
    return FEATURE_CACHE_DIR / f"{prefix}_{ym_to_label(ym)}.parquet"


def cleanup_feature_cache_paths(paths):
    for p in paths:
        try:
            Path(p).unlink(missing_ok=True)
        except PermissionError:
            pass


def cleanup_feature_cache(prefix: str):
    for p in FEATURE_CACHE_DIR.glob(f"{prefix}_*.parquet"):
        try:
            p.unlink()
        except PermissionError:
            pass


def build_many_to_parquet(target_yms, max_rows, prefix: str):
    cleanup_feature_cache(prefix)
    paths = []
    records = []
    cols = None
    rows_per_part = 0 if max_rows <= 0 else max(1, max_rows // len(target_yms))
    for ym in target_yms:
        print(f"building target {ym_to_label(ym)} ...")
        part = build_features(ym, include_target=True, skeleton_max_rows=rows_per_part, seed_offset=ym)
        before_second_sample = part.height
        part = sample_supervised(part, rows_per_part, seed_offset=ym)
        print(f"  rows: {before_second_sample:,}" + (f" -> sampled {part.height:,}" if part.height != before_second_sample else ""))
        if cols is None:
            cols = part.columns
        else:
            for c in cols:
                if c not in part.columns:
                    part = part.with_columns(pl.lit(0).alias(c))
            part = part.select(cols)
        path = _feature_cache_path(prefix, ym)
        part.write_parquet(path, compression="zstd")
        paths.append(path)
        records.append({"target_ym": ym, "target_month": ym_to_label(ym), "rows": part.height, "path": str(path)})
        del part
        gc.collect()
    return paths, cols, pl.DataFrame(records)


print("=" * 80)
print("BUILD TUNE TRAIN / VALIDATION")
print("=" * 80)
print(f"LOW_MEMORY_MODE: {LOW_MEMORY_MODE}")
print(f"Holiday features: {ENABLE_HOLIDAY_FEATURES}")
print(f"Caps tune/val/test/final: {MAX_TUNE_ROWS:,} / {MAX_VAL_ROWS:,} / {MAX_TEST_ROWS:,} / {MAX_FINAL_ROWS:,}")
train_feature_paths, all_cols, tune_build_stats = build_many_to_parquet(TRAIN_TARGET_YMS, MAX_TUNE_ROWS, "tune")
val_df = build_features(VAL_YM, include_target=True, skeleton_max_rows=MAX_VAL_ROWS, seed_offset=VAL_YM)
val_df = sample_supervised(val_df, MAX_VAL_ROWS, seed_offset=VAL_YM)

for c in all_cols:
    if c not in val_df.columns:
        val_df = val_df.with_columns(pl.lit(0).alias(c))
val_df = val_df.select(all_cols)

gc.collect()

print(f"train feature parts: {len(train_feature_paths)} files, {int(tune_build_stats['rows'].sum()):,} rows")
print(f"val_df             : {val_df.shape}")
print("test_df            : deferred until after tune training")
print("target distribution")
print(tune_build_stats.select(["target_month", "rows"]))


## Encoding

`location` is already an integer ID in the new dataset. It is kept as an integer categorical feature.

String categorical columns are fitted on train and reused for validation/test/blind.

In [ ]:
id_cols = ["location", "item_id"]
target_cols = ["y_qty"]
drop_cols = id_cols + target_cols

feature_cols = [c for c in all_cols if c not in drop_cols]
numeric_cat_cols = ["location", "target_month"]
string_cat_cols = [
    "item_id_code", "category_l1", "category_l2", "category_l3",
    "category", "brand", "manufacturer", "size"
]
string_cat_cols = [c for c in string_cat_cols if c in feature_cols]
cat_cols = [c for c in numeric_cat_cols if c in feature_cols] + string_cat_cols
num_cols = [c for c in feature_cols if c not in cat_cols]
cat_feature_indices = [feature_cols.index(c) for c in cat_cols]
MATRIX_CACHE_DIR = OUT / "_matrix_cache"
MEMMAP_MIN_ROWS = int(os.environ.get("MEMMAP_MIN_ROWS", "500000"))


def _scan_feature_paths(paths):
    return pl.scan_parquet([str(p) for p in paths])


def fit_category_maps_from_paths(paths, columns):
    maps = {}
    scan = _scan_feature_paths(paths)
    for col in columns:
        code_col = f"__{col}_code"
        mp = (
            scan.select(pl.col(col).cast(pl.Utf8).fill_null("__NA__").alias(col))
            .unique()
            .sort(col)
            .with_row_index(code_col)
            .with_columns(pl.col(code_col).cast(pl.Int32))
            .collect()
        )
        maps[col] = mp
    return maps


def fit_category_maps(df: pl.DataFrame, columns):
    maps = {}
    for col in columns:
        code_col = f"__{col}_code"
        mp = (
            df.select(pl.col(col).cast(pl.Utf8).fill_null("__NA__").alias(col))
            .unique()
            .sort(col)
            .with_row_index(code_col)
            .with_columns(pl.col(code_col).cast(pl.Int32))
        )
        maps[col] = mp
    return maps


def encode_string_categoricals(df: pl.DataFrame, maps, columns):
    out = df
    for col in columns:
        key_col = f"__{col}_key"
        code_col = f"__{col}_code"
        out = (
            out.with_columns(pl.col(col).cast(pl.Utf8).fill_null("__NA__").alias(key_col))
            .join(maps[col], left_on=key_col, right_on=col, how="left")
            .with_columns(pl.col(code_col).fill_null(-1).cast(pl.Int32).alias(col))
            .drop([key_col, code_col])
        )
    return out


def label_arrays_from_paths(paths):
    qty_parts = []
    for path in paths:
        y = pl.read_parquet(path, columns=["y_qty"])
        qty_parts.append(y["y_qty"].to_numpy().astype(np.float32, copy=False))
    return np.concatenate(qty_parts)


def _matrix_cache_path(name: str) -> Path:
    MATRIX_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    return MATRIX_CACHE_DIR / f"{name}.f32"


def _feature_exprs():
    exprs = []
    for col in feature_cols:
        if col in cat_cols:
            exprs.append(pl.col(col).fill_null(-1).cast(pl.Int32).cast(pl.Float32).alias(col))
        else:
            exprs.append(pl.col(col).fill_null(0).cast(pl.Float32).alias(col))
    return exprs


def _write_df_to_matrix(df: pl.DataFrame, mm, row_offset: int):
    exprs = _feature_exprs()
    chunk_rows = int(os.environ.get("MATRIX_CHUNK_ROWS", "200000"))
    for start in range(0, df.height, chunk_rows):
        end = min(start + chunk_rows, df.height)
        chunk = df.slice(start, end - start).select(exprs).to_numpy()
        mm[row_offset + start:row_offset + end, :] = np.asarray(chunk, dtype=np.float32, order="C")
        del chunk


def to_lgb_matrix(df: pl.DataFrame, name: str | None = None):
    n_rows = df.height
    n_cols = len(feature_cols)
    use_memmap = name is not None and n_rows >= MEMMAP_MIN_ROWS
    if not use_memmap:
        arr = df.select(_feature_exprs()).to_numpy()
        return np.asarray(arr, dtype=np.float32, order="C")

    path = _matrix_cache_path(name)
    mm = np.memmap(path, dtype=np.float32, mode="w+", shape=(n_rows, n_cols))
    _write_df_to_matrix(df, mm, 0)
    mm.flush()
    del mm
    return np.memmap(path, dtype=np.float32, mode="r", shape=(n_rows, n_cols))


def paths_to_lgb_matrix(paths, maps, name: str):
    n_rows = int(sum(pl.read_parquet(path, columns=["y_qty"]).height for path in paths))
    n_cols = len(feature_cols)
    path = _matrix_cache_path(name)
    mm = np.memmap(path, dtype=np.float32, mode="w+", shape=(n_rows, n_cols))
    row_offset = 0
    for part_path in paths:
        df = pl.read_parquet(part_path)
        for c in all_cols:
            if c not in df.columns:
                df = df.with_columns(pl.lit(0).alias(c))
        df = df.select(all_cols)
        df = encode_string_categoricals(df, maps, string_cat_cols)
        _write_df_to_matrix(df, mm, row_offset)
        row_offset += df.height
        del df
        gc.collect()
    mm.flush()
    del mm
    return np.memmap(path, dtype=np.float32, mode="r", shape=(n_rows, n_cols))


def cleanup_matrix_cache(*names: str):
    for name in names:
        path = MATRIX_CACHE_DIR / f"{name}.f32"
        try:
            if path.exists():
                path.unlink()
        except PermissionError:
            pass


cat_maps_tune = fit_category_maps_from_paths(train_feature_paths, string_cat_cols)

val_ids = val_df.select(id_cols).to_pandas(use_pyarrow_extension_array=False)
y_tune_qty = label_arrays_from_paths(train_feature_paths)
y_val_qty = val_df.select("y_qty").to_series().to_numpy().astype(np.float32, copy=False)

X_tune = paths_to_lgb_matrix(train_feature_paths, cat_maps_tune, "X_tune")
n_train_tune_rows = X_tune.shape[0]
cleanup_feature_cache_paths(train_feature_paths)
gc.collect()

val_df = encode_string_categoricals(val_df, cat_maps_tune, string_cat_cols)
X_val = to_lgb_matrix(val_df, "X_val")
n_val_rows = X_val.shape[0]
del val_df
gc.collect()

print("=" * 80)
print("FEATURE MATRIX")
print("=" * 80)
print(f"X_tune      : {X_tune.shape}")
print(f"X_val       : {X_val.shape}")
print("X_test      : deferred until after tune training")
print(f"Features    : {len(feature_cols):,}")
print(f"Categorical : {len(cat_cols):,} -> {cat_cols}")
print(f"Categorical indices: {cat_feature_indices}")
print(f"Numeric     : {len(num_cols):,}")


## Train Tune Model

Staged tuning is used to minimize validation `MAPE_quantity`:
- fixed anchor configs include the current best baseline, so tuning cannot select a worse tried config if Optuna fails;
- optional Optuna/TPE proposes additional quantity-weight configs, and optional narrow LightGBM params;
- the full-data candidate with the lowest validation MAPE after calibration scan is selected, then final calibration is recomputed and reported.


In [ ]:
def mape_nonzero(actual, pred):
    actual = np.asarray(actual, dtype=np.float64)
    pred = np.asarray(pred, dtype=np.float64)
    mask = actual > 0
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs(actual[mask] - pred[mask]) / actual[mask])


def wape(actual, pred):
    actual = np.asarray(actual, dtype=np.float64)
    pred = np.asarray(pred, dtype=np.float64)
    denom = np.sum(np.abs(actual))
    return np.nan if denom == 0 else np.sum(np.abs(actual - pred)) / denom


def compute_qty_metrics(actual_qty, pred_qty):
    actual_qty = np.asarray(actual_qty, dtype=np.float64)
    pred_qty = np.asarray(pred_qty, dtype=np.float64)
    return {
        "MAE_quantity": mean_absolute_error(actual_qty, pred_qty),
        "MAPE_quantity": mape_nonzero(actual_qty, pred_qty),
        "WAPE_quantity": wape(actual_qty, pred_qty),
        "bias_quantity": float(np.mean(pred_qty - actual_qty)),
        "positive_rows": int((actual_qty > 0).sum()),
        "zero_rows": int((actual_qty <= 0).sum()),
    }


def quantity_bucket_report(actual_qty, pred_qty):
    actual = np.asarray(actual_qty, dtype=np.float64)
    pred = np.asarray(pred_qty, dtype=np.float64)
    buckets = [
        ("zero", actual <= 0),
        ("qty_1", actual == 1),
        ("qty_2", actual == 2),
        ("qty_3", actual == 3),
        ("qty_4_5", (actual >= 4) & (actual <= 5)),
        ("qty_6_10", (actual >= 6) & (actual <= 10)),
        ("qty_11_25", (actual >= 11) & (actual <= 25)),
        ("qty_gt_25", actual > 25),
    ]
    rows = []
    for name, mask in buckets:
        if not mask.any():
            continue
        a = actual[mask]
        p = pred[mask]
        rows.append({
            "bucket": name,
            "rows": int(mask.sum()),
            "actual_mean": float(np.mean(a)),
            "pred_mean": float(np.mean(p)),
            "bias": float(np.mean(p - a)),
            "MAE_quantity": mean_absolute_error(a, p),
            "MAPE_quantity": mape_nonzero(a, p),
            "WAPE_quantity": wape(a, p),
        })
    return pd.DataFrame(rows)


TUNE_MODEL_PATH = ARTIFACT_DIR / "model_qty_tune.lgb"
TUNE_META_PATH = ARTIFACT_DIR / "tune_meta.json"
TUNE_RESULTS_PATH = ARTIFACT_DIR / "qty_weight_results.csv"
CALIB_META_PATH = ARTIFACT_DIR / "calibration_postprocess_meta.json"
POSTPROCESS_RESULTS_PATH = ARTIFACT_DIR / "qty_postprocess_results.csv"
FINAL_MODEL_PATH = ARTIFACT_DIR / "model_qty_final.lgb"
FINAL_META_PATH = ARTIFACT_DIR / "final_meta.json"
CAT_MAPS_FINAL_PATH = ARTIFACT_DIR / "cat_maps_final.pkl"
BLIND_IDS_PATH = ARTIFACT_DIR / "blind_ids.pkl"
BLIND_PRED_PATH = ARTIFACT_DIR / "pred_blind_qty.npy"


def _json_default(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    if isinstance(obj, Path):
        return str(obj)
    return str(obj)


def save_json(path, obj):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=_json_default), encoding="utf-8")
    tmp.replace(path)


def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def _artifact_signature():
    return {
        "experiment_version": EXPERIMENT_VERSION,
        "data_files": {"items": ITEMS_PATH.name, "transactions": TXN_PATH.name, "events": EVENT_PATH.name},
        "feature_cols": list(feature_cols),
        "cat_cols": list(cat_cols),
        "string_cat_cols": list(string_cat_cols),
        "train_target_yms": [int(x) for x in TRAIN_TARGET_YMS],
        "val_ym": int(VAL_YM),
        "test_ym": int(TEST_YM),
        "holiday_features": bool(ENABLE_HOLIDAY_FEATURES),
    }


def _signature_matches(meta):
    return meta.get("signature") == _artifact_signature()


def _parse_int_list(text, default):
    if text is None or str(text).strip() == "":
        return list(default)
    values = [int(part.strip()) for part in str(text).split(",") if part.strip()]
    return values or list(default)


QTY_SCALE_MIN = float(os.environ.get("QTY_SCALE_MIN", "0.75"))
QTY_SCALE_MAX = float(os.environ.get("QTY_SCALE_MAX", "1.40"))
QTY_SCALE_STEP = float(os.environ.get("QTY_SCALE_STEP", "0.01"))
QTY_SCALE_CANDIDATES = np.round(np.arange(QTY_SCALE_MIN, QTY_SCALE_MAX + 1e-9, QTY_SCALE_STEP), 3)
QTY_CALIBRATION_MODE = os.environ.get("QTY_CALIBRATION_MODE", "auto").lower()
QTY_BUCKET_COUNTS = _parse_int_list(os.environ.get("QTY_BUCKET_COUNTS", "4,5,6,8,10,12"), [6])
QTY_BUCKET_MIN_POS = int(os.environ.get("QTY_BUCKET_MIN_POS", "200"))
ENABLE_QTY_WEIGHT_SEARCH = os.environ.get("ENABLE_QTY_WEIGHT_SEARCH", "1") != "0"
QTY_WEIGHT_MAX_TRIALS = int(os.environ.get("QTY_WEIGHT_MAX_TRIALS", "4"))
QTY_TUNING_METHOD = os.environ.get("QTY_TUNING_METHOD", "staged_optuna").lower()
OPTUNA_WEIGHT_TRIALS = int(os.environ.get("OPTUNA_WEIGHT_TRIALS", "8"))
ENABLE_LGB_PARAM_TUNING = os.environ.get("ENABLE_LGB_PARAM_TUNING", "1") != "0"
ENABLE_QTY_ENSEMBLE = os.environ.get("ENABLE_QTY_ENSEMBLE", "1") != "0"
QTY_ENSEMBLE_CANDIDATE_POOL = int(os.environ.get("QTY_ENSEMBLE_CANDIDATE_POOL", "6"))
QTY_ENSEMBLE_TOP_K = int(os.environ.get("QTY_ENSEMBLE_TOP_K", "3"))
ENSEMBLE_WEIGHT_TRIALS = int(os.environ.get("ENSEMBLE_WEIGHT_TRIALS", "800"))
ENSEMBLE_WEIGHT_SAMPLE_ROWS = int(os.environ.get("ENSEMBLE_WEIGHT_SAMPLE_ROWS", "250000"))
ENSEMBLE_MIN_IMPROVEMENT = float(os.environ.get("ENSEMBLE_MIN_IMPROVEMENT", "0.0001"))
DEFAULT_QTY_WEIGHT_CONFIG = {
    "name": os.environ.get("QTY_WEIGHT_NAME", "inverse_sqrt_zero010"),
    "mode": os.environ.get("QTY_WEIGHT_MODE", "inverse_sqrt"),
    "zero_weight": float(os.environ.get("ZERO_QTY_WEIGHT", "0.10")),
    "clip_value": float(os.environ.get("QTY_WEIGHT_CLIP_VALUE", "10.0")),
    "pos_clip_min": float(os.environ.get("QTY_POS_WEIGHT_CLIP_MIN", "0.45")),
    "pos_clip_max": float(os.environ.get("QTY_POS_WEIGHT_CLIP_MAX", "3.5")),
    "weight_power": float(os.environ.get("QTY_WEIGHT_POWER", "0.50")),
    "small_qty_cutoff": float(os.environ.get("SMALL_QTY_CUTOFF", "3.0")),
    "small_qty_boost": float(os.environ.get("SMALL_QTY_BOOST", "1.55")),
    "tail_qty_cutoff": float(os.environ.get("TAIL_QTY_CUTOFF", "10.0")),
    "tail_qty_boost": float(os.environ.get("TAIL_QTY_BOOST", "0.75")),
}
QTY_WEIGHT_CANDIDATES = [
    {"name": "flat_zero010", "mode": "flat", "zero_weight": 0.10, "clip_value": 10.0, "pos_clip_min": 1.0, "pos_clip_max": 1.0},
    {"name": "small_tail_zero010", "mode": "small_tail", "zero_weight": 0.10, "clip_value": 10.0, "pos_clip_min": 0.45, "pos_clip_max": 3.5, "small_qty_cutoff": 3.0, "small_qty_boost": 1.55, "tail_qty_cutoff": 10.0, "tail_qty_boost": 0.75},
    {"name": "inverse_sqrt_zero010", "mode": "inverse_sqrt", "zero_weight": 0.10, "clip_value": 10.0, "pos_clip_min": 0.45, "pos_clip_max": 3.5},
    {"name": "inverse_sqrt_zero005", "mode": "inverse_sqrt", "zero_weight": 0.05, "clip_value": 10.0, "pos_clip_min": 0.35, "pos_clip_max": 4.0},
    {"name": "inverse_sqrt_zero015", "mode": "inverse_sqrt", "zero_weight": 0.15, "clip_value": 10.0, "pos_clip_min": 0.35, "pos_clip_max": 4.0},
    {"name": "power050_zero008", "mode": "power", "zero_weight": 0.08, "weight_power": 0.50, "pos_clip_min": 0.30, "pos_clip_max": 4.5},
    {"name": "power075_zero008", "mode": "power", "zero_weight": 0.08, "weight_power": 0.75, "pos_clip_min": 0.25, "pos_clip_max": 5.0},
    {"name": "small_tail_boost200_zero008", "mode": "small_tail", "zero_weight": 0.08, "clip_value": 10.0, "pos_clip_min": 0.35, "pos_clip_max": 4.5, "small_qty_cutoff": 3.0, "small_qty_boost": 2.0, "tail_qty_cutoff": 12.0, "tail_qty_boost": 0.55},
]
if not ENABLE_QTY_WEIGHT_SEARCH:
    QTY_WEIGHT_CANDIDATES_ACTIVE = [DEFAULT_QTY_WEIGHT_CONFIG]
elif QTY_WEIGHT_MAX_TRIALS > 0:
    QTY_WEIGHT_CANDIDATES_ACTIVE = QTY_WEIGHT_CANDIDATES[:QTY_WEIGHT_MAX_TRIALS]
else:
    QTY_WEIGHT_CANDIDATES_ACTIVE = QTY_WEIGHT_CANDIDATES


def _trial_weight_config(trial):
    mode = trial.suggest_categorical("weight_mode", ["inverse_sqrt", "power", "small_tail", "inverse_clip"])
    pos_clip_min = trial.suggest_float("pos_clip_min", 0.20, 0.80)
    pos_clip_max = trial.suggest_float("pos_clip_max", 2.0, 6.0)
    config = {
        "name": f"optuna_{trial.number:03d}_{mode}",
        "mode": mode,
        "zero_weight": trial.suggest_float("zero_weight", 0.03, 0.20, log=True),
        "pos_clip_min": min(pos_clip_min, pos_clip_max),
        "pos_clip_max": max(pos_clip_min, pos_clip_max),
    }
    if mode == "power":
        config["weight_power"] = trial.suggest_float("weight_power", 0.25, 1.00)
    elif mode == "small_tail":
        config.update({
            "small_qty_cutoff": trial.suggest_int("small_qty_cutoff", 1, 5),
            "small_qty_boost": trial.suggest_float("small_qty_boost", 0.8, 2.8),
            "tail_qty_cutoff": trial.suggest_float("tail_qty_cutoff", 6.0, 20.0),
            "tail_qty_boost": trial.suggest_float("tail_qty_boost", 0.2, 1.4),
        })
    elif mode == "inverse_clip":
        config["clip_value"] = trial.suggest_float("clip_value", 2.0, 20.0)
    return config


def _trial_params_qty(trial, base_params):
    params = dict(base_params)
    if ENABLE_LGB_PARAM_TUNING:
        params.update({
            "objective": trial.suggest_categorical("objective", ["regression_l1", "regression"]),
            "learning_rate": trial.suggest_float("learning_rate", 0.018, 0.070, log=True),
            "num_leaves": trial.suggest_categorical("num_leaves", [63, 95, 127, 191, 255]),
            "max_depth": trial.suggest_categorical("max_depth", [-1, 8, 10, 12]),
            "min_child_samples": trial.suggest_categorical("min_child_samples", [30, 50, 80, 120, 200, 350]),
            "feature_fraction": trial.suggest_float("feature_fraction", 0.65, 1.00),
            "bagging_fraction": trial.suggest_float("bagging_fraction", 0.65, 1.00),
            "bagging_freq": trial.suggest_categorical("bagging_freq", [1, 3, 5, 7]),
            "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 0.08),
            "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 5.0),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 12.0),
        })
    return params


def _best_global_qty_scale(y_true, pred_raw, scale_candidates=QTY_SCALE_CANDIDATES):
    rows = []
    for scale in scale_candidates:
        pred = np.clip(pred_raw * scale, 0, None)
        rows.append({
            "scale": float(scale),
            "MAPE_quantity": mape_nonzero(y_true, pred),
            "MAE_quantity": mean_absolute_error(y_true, pred),
            "WAPE_quantity": wape(y_true, pred),
            "bias_quantity": float(np.mean(pred - y_true)),
        })
    return pd.DataFrame(rows).sort_values(["MAPE_quantity", "MAE_quantity"]).reset_index(drop=True)


def _best_candidate_calibration_score(y_true, pred_raw):
    global_table = _best_global_qty_scale(y_true, pred_raw)
    global_scale = float(global_table.loc[0, "scale"])
    best_pred = np.clip(pred_raw * global_scale, 0, None)
    best = {
        "score_calibration_mode": "global",
        "score_bucket_count": 0,
        "score_scale": global_scale,
        "score_MAPE_quantity": mape_nonzero(y_true, best_pred),
        "score_MAE_quantity": mean_absolute_error(y_true, best_pred),
        "score_WAPE_quantity": wape(y_true, best_pred),
        "score_bias_quantity": float(np.mean(best_pred - y_true)),
    }
    if QTY_CALIBRATION_MODE == "global":
        return best, global_table

    finite_pred = pred_raw[np.isfinite(pred_raw)]
    if finite_pred.size == 0:
        return best, global_table

    for bucket_count in QTY_BUCKET_COUNTS:
        edges = np.unique(np.quantile(finite_pred, np.linspace(0.0, 1.0, int(bucket_count) + 1)))
        if edges.size < 3:
            continue
        edges[0] = -np.inf
        edges[-1] = np.inf
        bucket_idx = np.searchsorted(edges[1:-1], pred_raw, side="right")
        pred_bucket = np.empty_like(pred_raw, dtype=np.float64)
        for bucket in range(edges.size - 1):
            mask = bucket_idx == bucket
            if not mask.any():
                continue
            pos_count = int(((y_true > 0) & mask).sum())
            if pos_count < QTY_BUCKET_MIN_POS:
                scale = global_scale
            else:
                scale = float(_best_global_qty_scale(y_true[mask], pred_raw[mask]).loc[0, "scale"])
            pred_bucket[mask] = np.clip(pred_raw[mask] * scale, 0, None)
        row = {
            "score_calibration_mode": "pred_bucket",
            "score_bucket_count": int(bucket_count),
            "score_scale": global_scale,
            "score_MAPE_quantity": mape_nonzero(y_true, pred_bucket),
            "score_MAE_quantity": mean_absolute_error(y_true, pred_bucket),
            "score_WAPE_quantity": wape(y_true, pred_bucket),
            "score_bias_quantity": float(np.mean(pred_bucket - y_true)),
        }
        if np.isfinite(row["score_MAPE_quantity"]) and row["score_MAPE_quantity"] < best["score_MAPE_quantity"]:
            best = row
    return best, global_table


def make_quantity_weights(y_qty, config=None):
    config = config or globals().get("selected_qty_weight_config", DEFAULT_QTY_WEIGHT_CONFIG)
    y = np.asarray(y_qty, dtype=np.float64)
    zero_weight = float(config.get("zero_weight", 0.10))
    weights = np.full(y.shape[0], zero_weight, dtype=np.float64)
    pos = y > 0
    if pos.sum() == 0:
        return weights.astype(np.float32)

    mode = config.get("mode", "small_tail")
    y_pos = np.maximum(y[pos], 1.0)
    if mode == "flat":
        pos_w = np.ones_like(y_pos, dtype=np.float64)
    elif mode == "inverse_sqrt":
        pos_w = 1.0 / np.sqrt(y_pos)
    elif mode == "power":
        power = float(config.get("weight_power", 0.50))
        pos_w = 1.0 / np.power(y_pos, power)
    elif mode == "inverse_clip":
        clip_value = float(config.get("clip_value", 10.0))
        pos_w = 1.0 / np.clip(y_pos, 1.0, clip_value)
    elif mode == "tail_guard":
        cutoff = float(config.get("tail_qty_cutoff", 8.0))
        boost = float(config.get("tail_qty_boost", 1.25))
        pos_w = 1.0 + boost * np.log1p(np.maximum(y_pos - cutoff, 0.0)) / np.log1p(np.maximum(y_pos.max() - cutoff, 1.0))
    else:
        small_cutoff = float(config.get("small_qty_cutoff", 3.0))
        small_boost = float(config.get("small_qty_boost", 1.55))
        tail_cutoff = float(config.get("tail_qty_cutoff", 10.0))
        tail_boost = float(config.get("tail_qty_boost", 0.75))
        pos_w = np.ones_like(y_pos, dtype=np.float64)
        pos_w += (y_pos <= small_cutoff) * small_boost
        tail_denom = np.log1p(np.maximum(y_pos.max() - tail_cutoff, 1.0))
        pos_w += tail_boost * np.log1p(np.maximum(y_pos - tail_cutoff, 0.0)) / tail_denom

    mean_w = float(np.mean(pos_w))
    if np.isfinite(mean_w) and mean_w > 0:
        pos_w = pos_w / mean_w
    clip_min = float(config.get("pos_clip_min", 0.45))
    clip_max = float(config.get("pos_clip_max", 3.5))
    if clip_max < clip_min:
        clip_min, clip_max = clip_max, clip_min
    pos_w = np.clip(pos_w, clip_min, clip_max)
    weights[pos] = pos_w
    return weights.astype(np.float32)


def feval_log_mape_quantity(preds, dataset):
    actual = np.expm1(dataset.get_label())
    pred = np.clip(np.expm1(preds), 0, None)
    return "mape_quantity", mape_nonzero(actual, pred), False


params_qty = {
    "boosting_type": "gbdt",
    "objective": "regression_l1",
    "learning_rate": float(os.environ.get("LGB_LEARNING_RATE", "0.04")),
    "num_leaves": int(os.environ.get("LGB_NUM_LEAVES", "95")),
    "max_depth": int(os.environ.get("LGB_MAX_DEPTH", "-1")),
    "min_child_samples": int(os.environ.get("LGB_MIN_CHILD_SAMPLES", "60")),
    "feature_fraction": float(os.environ.get("LGB_FEATURE_FRACTION", "0.85")),
    "bagging_fraction": float(os.environ.get("LGB_BAGGING_FRACTION", "0.85")),
    "bagging_freq": int(os.environ.get("LGB_BAGGING_FREQ", "5")),
    "min_split_gain": float(os.environ.get("LGB_MIN_SPLIT_GAIN", "0.0")),
    "reg_alpha": float(os.environ.get("LGB_REG_ALPHA", "0.2")),
    "reg_lambda": float(os.environ.get("LGB_REG_LAMBDA", "0.8")),
    "max_bin": int(os.environ.get("LGB_MAX_BIN", "127")),
    "metric": "None",
    "verbose": -1,
    "seed": SEED,
    "num_threads": N_THREADS,
    "force_col_wise": True,
}


def _train_qty_tune_for_weight(config):
    candidate_params = dict(config.get("_params_qty", params_qty))
    dtrain_qty = lgb.Dataset(
        X_tune,
        label=np.log1p(y_tune_qty),
        weight=make_quantity_weights(y_tune_qty, config),
        feature_name=feature_cols,
        categorical_feature=cat_feature_indices,
        free_raw_data=True,
    )
    dval_qty = lgb.Dataset(
        X_val,
        label=np.log1p(y_val_qty),
        reference=dtrain_qty,
        feature_name=feature_cols,
        categorical_feature=cat_feature_indices,
        free_raw_data=True,
    )
    print("\n" + "=" * 80)
    print(f"TRAIN TUNE - QUANTITY - {config['name']}")
    print("=" * 80)
    model = lgb.train(
        candidate_params,
        dtrain_qty,
        num_boost_round=int(os.environ.get("LGB_NUM_BOOST_ROUND", "1800")),
        valid_sets=[dtrain_qty, dval_qty],
        valid_names=["train", "val_nov"],
        feval=feval_log_mape_quantity,
        callbacks=[lgb.log_evaluation(100), lgb.early_stopping(80, first_metric_only=True)],
    )
    best_iter = model.best_iteration
    pred_raw = np.clip(np.expm1(model.predict(X_val, num_iteration=best_iter)), 0, None)
    score_info, scale_df = _best_candidate_calibration_score(y_val_qty, pred_raw)
    best_scale = float(scale_df.loc[0, "scale"])
    pred_scaled = np.clip(pred_raw * best_scale, 0, None)
    result = {
        "name": config["name"],
        "mode": config.get("mode"),
        "zero_weight": float(config.get("zero_weight", np.nan)),
        "best_iteration": int(best_iter),
        "raw_MAPE_quantity": mape_nonzero(y_val_qty, pred_raw),
        "raw_MAE_quantity": mean_absolute_error(y_val_qty, pred_raw),
        "global_scale": best_scale,
        "scaled_MAPE_quantity": mape_nonzero(y_val_qty, pred_scaled),
        "scaled_MAE_quantity": mean_absolute_error(y_val_qty, pred_scaled),
        "scaled_WAPE_quantity": wape(y_val_qty, pred_scaled),
        "scaled_bias_quantity": float(np.mean(pred_scaled - y_val_qty)),
        **score_info,
        "lgb_objective": str(candidate_params.get("objective", "")),
        "lgb_learning_rate": float(candidate_params.get("learning_rate", np.nan)),
        "lgb_num_leaves": int(candidate_params.get("num_leaves", -1)),
        "lgb_max_depth": int(candidate_params.get("max_depth", -1)),
        "lgb_min_child_samples": int(candidate_params.get("min_child_samples", -1)),
        "lgb_feature_fraction": float(candidate_params.get("feature_fraction", np.nan)),
        "lgb_bagging_fraction": float(candidate_params.get("bagging_fraction", np.nan)),
        "lgb_bagging_freq": int(candidate_params.get("bagging_freq", 0)),
        "lgb_min_split_gain": float(candidate_params.get("min_split_gain", np.nan)),
        "lgb_reg_alpha": float(candidate_params.get("reg_alpha", np.nan)),
        "lgb_reg_lambda": float(candidate_params.get("reg_lambda", np.nan)),
    }
    del dtrain_qty, dval_qty, pred_raw, pred_scaled
    gc.collect()
    return model, best_iter, result


qty_weight_rows = []
qty_candidate_registry = []
model_qty_tune = None
best_iter_qty = None
selected_qty_weight_config = None
selected_params_qty = dict(params_qty)
best_qty_weight_score = np.inf
QTY_TUNING_METHOD_EFFECTIVE = "anchor_full_search"
loaded_tune_artifact = False


def _save_tune_checkpoint(best_result=None):
    if model_qty_tune is None:
        return
    model_qty_tune.save_model(str(TUNE_MODEL_PATH))
    meta = {
        "signature": _artifact_signature(),
        "selected_qty_weight_config": selected_qty_weight_config,
        "selected_params_qty": selected_params_qty,
        "best_iter_qty": int(best_iter_qty),
        "best_qty_weight_score": float(best_qty_weight_score),
        "tuning_method_effective": QTY_TUNING_METHOD_EFFECTIVE,
        "best_result": best_result or {},
        "candidate_registry": qty_candidate_registry,
    }
    save_json(TUNE_META_PATH, meta)
    if qty_weight_rows:
        pd.DataFrame(qty_weight_rows).to_csv(TUNE_RESULTS_PATH, index=False)


def _load_tune_checkpoint():
    global model_qty_tune, best_iter_qty, selected_qty_weight_config, selected_params_qty
    global best_qty_weight_score, QTY_TUNING_METHOD_EFFECTIVE, params_qty, qty_weight_results, qty_candidate_registry
    if not (RESUME_ARTIFACTS and not FORCE_RETRAIN and TUNE_MODEL_PATH.exists() and TUNE_META_PATH.exists()):
        return False
    meta = load_json(TUNE_META_PATH)
    if not _signature_matches(meta):
        print("Tune artifact exists but signature changed; retraining tune model.")
        return False
    model_qty_tune = lgb.Booster(model_file=str(TUNE_MODEL_PATH))
    selected_qty_weight_config = dict(meta["selected_qty_weight_config"])
    selected_params_qty = dict(meta["selected_params_qty"])
    params_qty = dict(selected_params_qty)
    best_iter_qty = int(meta["best_iter_qty"])
    best_qty_weight_score = float(meta.get("best_qty_weight_score", np.inf))
    QTY_TUNING_METHOD_EFFECTIVE = str(meta.get("tuning_method_effective", "loaded_tune_artifact"))
    qty_candidate_registry = list(meta.get("candidate_registry", []))
    if TUNE_RESULTS_PATH.exists():
        qty_weight_results = pd.read_csv(TUNE_RESULTS_PATH)
    else:
        qty_weight_results = pd.DataFrame([meta.get("best_result", {})])
    print("=" * 80)
    print("LOADED TUNE MODEL ARTIFACT")
    print("=" * 80)
    print(TUNE_MODEL_PATH)
    print(f"Selected quantity weight    : {selected_qty_weight_config}")
    print(f"Selected quantity best iter : {best_iter_qty}")
    print(f"Saved validation MAPE       : {best_qty_weight_score:.6f}")
    display(qty_weight_results)
    return True


def _safe_name(text):
    return "".join(ch if ch.isalnum() or ch in {"_", "-"} else "_" for ch in str(text))[:80]


def _consider_candidate(config, source):
    global model_qty_tune, best_iter_qty, selected_qty_weight_config, selected_params_qty, best_qty_weight_score
    candidate_model, candidate_iter, result = _train_qty_tune_for_weight(config)
    result["source"] = source
    candidate_idx = len(qty_candidate_registry)
    candidate_path = TUNE_CANDIDATE_DIR / f"candidate_{candidate_idx:03d}_{_safe_name(config.get('name', source))}.lgb"
    candidate_model.save_model(str(candidate_path))
    result["candidate_index"] = int(candidate_idx)
    result["candidate_model_path"] = str(candidate_path)
    stored_config = {k: v for k, v in config.items() if not k.startswith("_")}
    stored_params = dict(config.get("_params_qty", params_qty))
    qty_candidate_registry.append({
        "candidate_index": int(candidate_idx),
        "source": source,
        "model_path": str(candidate_path),
        "best_iteration": int(candidate_iter),
        "weight_config": stored_config,
        "params_qty": stored_params,
        "result": dict(result),
    })
    qty_weight_rows.append(result)
    score = result["score_MAPE_quantity"]
    if np.isfinite(score) and score < best_qty_weight_score:
        if model_qty_tune is not None:
            del model_qty_tune
            gc.collect()
        model_qty_tune = candidate_model
        best_iter_qty = candidate_iter
        selected_qty_weight_config = stored_config
        selected_params_qty = stored_params
        best_qty_weight_score = score
        _save_tune_checkpoint(result)
    else:
        del candidate_model
        gc.collect()
    return score


loaded_tune_artifact = _load_tune_checkpoint()

if not loaded_tune_artifact:
    print("=" * 80)
    print("QUANTITY STAGED TUNING - VALIDATION MAPE FOCUS")
    print("=" * 80)
    print(f"Tuning method requested : {QTY_TUNING_METHOD}")
    print(f"Scale candidates        : {QTY_SCALE_CANDIDATES[0]} -> {QTY_SCALE_CANDIDATES[-1]} step {QTY_SCALE_STEP}")
    print(f"Calibration bucket scan : {QTY_BUCKET_COUNTS}")
    print(f"LGB param tuning        : {ENABLE_LGB_PARAM_TUNING}")
    print(f"Ensemble enabled        : {ENABLE_QTY_ENSEMBLE} / top_k={QTY_ENSEMBLE_TOP_K} / pool={QTY_ENSEMBLE_CANDIDATE_POOL}")
    print(f"Anchor candidates       : {[c['name'] for c in QTY_WEIGHT_CANDIDATES_ACTIVE]}")

    for config in QTY_WEIGHT_CANDIDATES_ACTIVE:
        _consider_candidate(dict(config), "anchor")

    try:
        import optuna
        optuna_available = True
    except ImportError:
        optuna = None
        optuna_available = False

    if QTY_TUNING_METHOD in {"staged_optuna", "optuna", "optuna_tpe"} and OPTUNA_WEIGHT_TRIALS > 0:
        if not optuna_available:
            print("Optuna is not installed; anchor candidates were used. Install optuna to enable TPE search.")
        else:
            QTY_TUNING_METHOD_EFFECTIVE = "anchor_plus_optuna_tpe"
            sampler = optuna.samplers.TPESampler(seed=SEED, multivariate=True, group=True)
            study = optuna.create_study(direction="minimize", sampler=sampler)
            for _ in range(OPTUNA_WEIGHT_TRIALS):
                trial = study.ask()
                config = _trial_weight_config(trial)
                config["_params_qty"] = _trial_params_qty(trial, params_qty)
                score = _consider_candidate(config, "optuna_tpe")
                study.tell(trial, score)

    params_qty = dict(selected_params_qty)
    qty_weight_results = pd.DataFrame(qty_weight_rows).sort_values(["score_MAPE_quantity", "scaled_MAPE_quantity", "scaled_MAE_quantity"]).reset_index(drop=True)
    qty_weight_results.to_csv(TUNE_RESULTS_PATH, index=False)
    _save_tune_checkpoint(qty_weight_results.iloc[0].to_dict() if len(qty_weight_results) else {})
    display(qty_weight_results)
    print(f"Effective tuning method       : {QTY_TUNING_METHOD_EFFECTIVE}")
    print(f"Selected quantity weight      : {selected_qty_weight_config}")
    print(f"Selected quantity best iter   : {best_iter_qty}")
    print(f"Selected validation MAPE      : {best_qty_weight_score:.6f}")
    print("Selected LightGBM params      :", {k: params_qty.get(k) for k in ["objective", "learning_rate", "num_leaves", "max_depth", "min_child_samples", "feature_fraction", "bagging_fraction", "bagging_freq", "min_split_gain", "reg_alpha", "reg_lambda"]})

del X_tune, y_tune_qty
cleanup_matrix_cache("X_tune")
gc.collect()


In [ ]:
pred_val_qty_raw = np.clip(np.expm1(model_qty_tune.predict(X_val, num_iteration=best_iter_qty)), 0, None)
# Keep X_val until ensemble evaluation; cleaned up near the end of this cell.

scale_candidates = QTY_SCALE_CANDIDATES
QTY_CALIBRATION_MODE = os.environ.get("QTY_CALIBRATION_MODE", "auto").lower()
QTY_BUCKET_COUNTS = _parse_int_list(os.environ.get("QTY_BUCKET_COUNTS", ",".join(map(str, QTY_BUCKET_COUNTS))), QTY_BUCKET_COUNTS)
QTY_BUCKET_MIN_POS = int(os.environ.get("QTY_BUCKET_MIN_POS", "200"))
ENABLE_QTY_POSTPROCESS = os.environ.get("ENABLE_QTY_POSTPROCESS", "1") != "0"
POSTPROCESS_FIT_SPLIT = os.environ.get("POSTPROCESS_FIT_SPLIT", "val_test_avg").lower()


def _fit_global_qty_calibrator(y_true, pred_raw):
    table = _best_global_qty_scale(y_true, pred_raw, scale_candidates)
    scale = float(table.loc[0, "scale"])
    calibrator = {"mode": "global", "scale": scale, "fallback_scale": scale, "bucket_count": 0}
    return calibrator, table


def _fit_pred_bucket_qty_calibrator(y_true, pred_raw, fallback_scale, bucket_count):
    y_true = np.asarray(y_true, dtype=np.float64)
    pred_raw = np.asarray(pred_raw, dtype=np.float64)
    finite_pred = pred_raw[np.isfinite(pred_raw)]
    if finite_pred.size == 0:
        return {"mode": "global", "scale": fallback_scale, "fallback_scale": fallback_scale, "bucket_count": 0}, pd.DataFrame()

    quantiles = np.linspace(0.0, 1.0, int(bucket_count) + 1)
    edges = np.quantile(finite_pred, quantiles)
    edges = np.unique(edges)
    if edges.size < 3:
        return {"mode": "global", "scale": fallback_scale, "fallback_scale": fallback_scale, "bucket_count": 0}, pd.DataFrame()

    edges[0] = -np.inf
    edges[-1] = np.inf
    bucket_idx = np.searchsorted(edges[1:-1], pred_raw, side="right")
    scales = []
    rows = []
    for bucket in range(edges.size - 1):
        mask = bucket_idx == bucket
        pos_count = int(((y_true > 0) & mask).sum())
        row = {
            "bucket_count": int(bucket_count),
            "bucket": bucket,
            "left": float(edges[bucket]) if np.isfinite(edges[bucket]) else -np.inf,
            "right": float(edges[bucket + 1]) if np.isfinite(edges[bucket + 1]) else np.inf,
            "rows": int(mask.sum()),
            "positive_rows": pos_count,
        }
        if pos_count < QTY_BUCKET_MIN_POS:
            scale = fallback_scale
            pred = np.clip(pred_raw[mask] * scale, 0, None)
            row.update({"scale": scale, "MAPE_quantity": mape_nonzero(y_true[mask], pred) if mask.any() else np.nan, "status": "fallback_low_positive_rows"})
        else:
            scale_table = _best_global_qty_scale(y_true[mask], pred_raw[mask], scale_candidates)
            scale = float(scale_table.loc[0, "scale"])
            row.update({"scale": scale, "MAPE_quantity": float(scale_table.loc[0, "MAPE_quantity"]), "status": "fitted"})
        scales.append(scale)
        rows.append(row)

    calibrator = {
        "mode": "pred_bucket",
        "bucket_count": int(bucket_count),
        "edges": [float(x) if np.isfinite(x) else ("-inf" if x < 0 else "inf") for x in edges],
        "scales": [float(x) for x in scales],
        "fallback_scale": float(fallback_scale),
        "bucket_min_pos": int(QTY_BUCKET_MIN_POS),
    }
    return calibrator, pd.DataFrame(rows)


def apply_qty_calibration(pred_raw, calibrator):
    pred_raw = np.asarray(pred_raw, dtype=np.float64)
    mode = calibrator.get("mode", "global")
    if mode == "pred_bucket":
        raw_edges = calibrator.get("edges", [])
        edges = np.array([-np.inf if x == "-inf" else np.inf if x == "inf" else float(x) for x in raw_edges], dtype=np.float64)
        scales = np.asarray(calibrator.get("scales", []), dtype=np.float64)
        if edges.size >= 2 and scales.size == edges.size - 1:
            bucket_idx = np.searchsorted(edges[1:-1], pred_raw, side="right")
            bucket_idx = np.clip(bucket_idx, 0, scales.size - 1)
            return np.clip(pred_raw * scales[bucket_idx], 0, None)
    return np.clip(pred_raw * float(calibrator.get("scale", calibrator.get("fallback_scale", 1.0))), 0, None)


def apply_qty_postprocess(pred, config):
    pred = np.asarray(pred, dtype=np.float64).copy()
    if not config or config.get("name") == "none":
        return np.clip(pred, 0, None)
    pred *= float(config.get("global_scale", 1.0))
    high_threshold = config.get("high_threshold")
    high_scale = float(config.get("high_scale", 1.0))
    if high_threshold is not None and high_scale != 1.0:
        mask = pred >= float(high_threshold)
        pred[mask] *= high_scale
    snap_low = config.get("snap_low")
    snap_high = config.get("snap_high")
    if snap_low is not None and snap_high is not None:
        mask = (pred >= float(snap_low)) & (pred <= float(snap_high))
        pred[mask] = 1.0
    cap = config.get("cap")
    if cap is not None:
        pred = np.minimum(pred, float(cap))
    floor = config.get("floor")
    if floor is not None:
        pred = np.maximum(pred, float(floor))
    return np.clip(pred, 0, None)


def _postprocess_candidates():
    if not ENABLE_QTY_POSTPROCESS:
        return [{"name": "none"}]
    candidates = [{"name": "none"}]
    global_scales = [0.98, 1.0, 1.02]
    snap_bands = [(None, None), (0.85, 1.15), (0.80, 1.20), (0.75, 1.25), (0.90, 1.10)]
    caps = [None, 10.0, 12.0, 14.0]
    high_settings = [(None, 1.0), (5.0, 0.95), (5.0, 0.90), (8.0, 0.95), (8.0, 0.90), (10.0, 0.90), (10.0, 0.85)]
    for global_scale in global_scales:
        for snap_low, snap_high in snap_bands:
            for cap in caps:
                for high_threshold, high_scale in high_settings:
                    if snap_low is None and cap is None and high_threshold is None and global_scale == 1.0:
                        continue
                    cfg = {
                        "name": "post",
                        "global_scale": global_scale,
                        "snap_low": snap_low,
                        "snap_high": snap_high,
                        "cap": cap,
                        "high_threshold": high_threshold,
                        "high_scale": high_scale,
                    }
                    cfg["name"] = "post_g{:.2f}_snap{}_{}_cap{}_hi{}_{}".format(
                        global_scale,
                        "none" if snap_low is None else snap_low,
                        "none" if snap_high is None else snap_high,
                        "none" if cap is None else cap,
                        "none" if high_threshold is None else high_threshold,
                        high_scale,
                    )
                    candidates.append(cfg)
    return candidates


def fit_qty_postprocess(y_true, pred_calibrated):
    rows = []
    candidates = _postprocess_candidates()
    for cfg in candidates:
        pred = apply_qty_postprocess(pred_calibrated, cfg)
        rows.append({
            **cfg,
            "MAPE_quantity": mape_nonzero(y_true, pred),
            "MAE_quantity": mean_absolute_error(y_true, pred),
            "WAPE_quantity": wape(y_true, pred),
            "bias_quantity": float(np.mean(pred - y_true)),
        })
    eval_df = pd.DataFrame(rows).sort_values(["MAPE_quantity", "MAE_quantity"]).reset_index(drop=True)
    selected = eval_df.iloc[0].where(pd.notna(eval_df.iloc[0]), None).to_dict()
    selected = {k: (None if pd.isna(v) else v) for k, v in selected.items()}
    return selected, eval_df


def fit_qty_postprocess_multi(splits):
    rows = []
    candidates = _postprocess_candidates()
    for cfg in candidates:
        row = {**cfg}
        mape_values = []
        mae_values = []
        for split_name, y_true, pred_calibrated in splits:
            pred = apply_qty_postprocess(pred_calibrated, cfg)
            mape_v = mape_nonzero(y_true, pred)
            mae_v = mean_absolute_error(y_true, pred)
            row[f"{split_name}_MAPE_quantity"] = mape_v
            row[f"{split_name}_MAE_quantity"] = mae_v
            row[f"{split_name}_WAPE_quantity"] = wape(y_true, pred)
            row[f"{split_name}_bias_quantity"] = float(np.mean(pred - y_true))
            mape_values.append(mape_v)
            mae_values.append(mae_v)
        row["MAPE_quantity"] = float(np.nanmean(mape_values))
        row["MAE_quantity"] = float(np.nanmean(mae_values))
        rows.append(row)
    eval_df = pd.DataFrame(rows).sort_values(["MAPE_quantity", "MAE_quantity"]).reset_index(drop=True)
    selected = eval_df.iloc[0].where(pd.notna(eval_df.iloc[0]), None).to_dict()
    selected = {k: (None if pd.isna(v) else v) for k, v in selected.items()}
    return selected, eval_df


def _fit_calibration_and_postprocess():
    global_qty_calibrator, qty_scale_df_local = _fit_global_qty_calibrator(y_val_qty, pred_val_qty_raw)
    calibration_options = [("global_0", "global", global_qty_calibrator)]
    bucket_tables = {}
    if QTY_CALIBRATION_MODE != "global":
        for bucket_count in QTY_BUCKET_COUNTS:
            bucket_qty_calibrator, bucket_df = _fit_pred_bucket_qty_calibrator(
                y_val_qty, pred_val_qty_raw, global_qty_calibrator["scale"], bucket_count
            )
            if bucket_qty_calibrator.get("mode") == "pred_bucket":
                option_key = f"pred_bucket_{bucket_count}"
                calibration_options.append((option_key, "pred_bucket", bucket_qty_calibrator))
                bucket_tables[option_key] = bucket_df

    allowed_options = []
    for option_key, name, calibrator in calibration_options:
        if QTY_CALIBRATION_MODE == "global" and name != "global":
            continue
        if QTY_CALIBRATION_MODE == "pred_bucket" and name != "pred_bucket":
            continue
        allowed_options.append((option_key, name, calibrator))
    if not allowed_options:
        allowed_options = [("global_0", "global", global_qty_calibrator)]

    rows = []
    for option_key, name, calibrator in allowed_options:
        pred = apply_qty_calibration(pred_val_qty_raw, calibrator)
        rows.append({
            "option_key": option_key,
            "mode": name,
            "bucket_count": int(calibrator.get("bucket_count", 0)),
            "scale": float(calibrator.get("scale", calibrator.get("fallback_scale", 1.0))),
            "MAPE_quantity": mape_nonzero(y_val_qty, pred),
            "MAE_quantity": mean_absolute_error(y_val_qty, pred),
            "WAPE_quantity": wape(y_val_qty, pred),
            "bias_quantity": float(np.mean(pred - y_val_qty)),
        })
    calibration_eval = pd.DataFrame(rows).sort_values(["MAPE_quantity", "MAE_quantity"]).reset_index(drop=True)
    selected_key = str(calibration_eval.loc[0, "option_key"])
    calibrator = dict([x for x in allowed_options if x[0] == selected_key][0][2])
    bucket_df = bucket_tables.get(selected_key, pd.DataFrame())
    pred_calibrated = apply_qty_calibration(pred_val_qty_raw, calibrator)
    postprocess_config, postprocess_eval = fit_qty_postprocess(y_val_qty, pred_calibrated)
    return calibrator, postprocess_config, calibration_eval, qty_scale_df_local, bucket_df, postprocess_eval


loaded_calibration_artifact = False
if RESUME_ARTIFACTS and not FORCE_RETRAIN and CALIB_META_PATH.exists():
    meta = load_json(CALIB_META_PATH)
    if _signature_matches(meta) and meta.get("best_iter_qty") == int(best_iter_qty) and meta.get("postprocess_fit_split", "val") == POSTPROCESS_FIT_SPLIT:
        selected_qty_calibrator = dict(meta["selected_qty_calibrator"])
        selected_qty_postprocess = dict(meta.get("selected_qty_postprocess", {"name": "none"}))
        qty_calibration_eval = pd.DataFrame(meta.get("qty_calibration_eval", []))
        qty_scale_df = pd.DataFrame(meta.get("qty_scale_df", []))
        qty_bucket_scale_df = pd.DataFrame(meta.get("qty_bucket_scale_df", []))
        qty_postprocess_eval = pd.read_csv(POSTPROCESS_RESULTS_PATH) if POSTPROCESS_RESULTS_PATH.exists() else pd.DataFrame(meta.get("qty_postprocess_eval", []))
        loaded_calibration_artifact = True
        print("=" * 80)
        print("LOADED CALIBRATION / POSTPROCESS ARTIFACT")
        print("=" * 80)
        print(CALIB_META_PATH)
    else:
        print("Calibration artifact exists but signature/iteration changed; refitting calibration.")

if not loaded_calibration_artifact:
    selected_qty_calibrator, selected_qty_postprocess, qty_calibration_eval, qty_scale_df, qty_bucket_scale_df, qty_postprocess_eval = _fit_calibration_and_postprocess()
    qty_postprocess_eval.to_csv(POSTPROCESS_RESULTS_PATH, index=False)
    save_json(CALIB_META_PATH, {
        "signature": _artifact_signature(),
        "best_iter_qty": int(best_iter_qty),
        "selected_qty_calibrator": selected_qty_calibrator,
        "selected_qty_postprocess": selected_qty_postprocess,
        "qty_calibration_eval": qty_calibration_eval.to_dict(orient="records"),
        "qty_scale_df": qty_scale_df.to_dict(orient="records"),
        "qty_bucket_scale_df": qty_bucket_scale_df.to_dict(orient="records"),
        "qty_postprocess_eval": qty_postprocess_eval.head(50).to_dict(orient="records"),
        "postprocess_fit_split": "val",
    })

best_qty_scale = float(selected_qty_calibrator.get("scale", selected_qty_calibrator.get("fallback_scale", 1.0)))
pred_val_qty_calibrated = apply_qty_calibration(pred_val_qty_raw, selected_qty_calibrator)
pred_val_qty = apply_qty_postprocess(pred_val_qty_calibrated, selected_qty_postprocess)
metrics_val_raw = compute_qty_metrics(y_val_qty, pred_val_qty_raw)
metrics_val_calibrated_only = compute_qty_metrics(y_val_qty, pred_val_qty_calibrated)
metrics_val = compute_qty_metrics(y_val_qty, pred_val_qty)
val_bucket_raw = quantity_bucket_report(y_val_qty, pred_val_qty_raw)
val_bucket = quantity_bucket_report(y_val_qty, pred_val_qty)

print("=" * 80)
print("VALIDATION NOVEMBER 2025 - QUANTITY ONLY")
print("=" * 80)
print(f"Quantity weight config : {selected_qty_weight_config}")
print(f"Quantity calibration   : {selected_qty_calibrator.get('mode')} / bucket {selected_qty_calibrator.get('bucket_count', 0)} / fallback scale {best_qty_scale:.3f}")
print(f"Quantity postprocess   : {selected_qty_postprocess.get('name')}")
display(qty_calibration_eval)
if not qty_bucket_scale_df.empty:
    display(qty_bucket_scale_df)
display(qty_postprocess_eval.head(20))
print("Raw:", metrics_val_raw)
print("Calibrated only:", metrics_val_calibrated_only)
print("Calibrated + postprocess:", metrics_val)
display(val_bucket)

print("\n" + "=" * 80)
print("BUILD INTERNAL TEST FEATURES")
print("=" * 80)
test_df = build_features(TEST_YM, include_target=True, skeleton_max_rows=MAX_TEST_ROWS, seed_offset=TEST_YM)
test_df = sample_supervised(test_df, MAX_TEST_ROWS, seed_offset=TEST_YM)
for c in all_cols:
    if c not in test_df.columns:
        test_df = test_df.with_columns(pl.lit(0).alias(c))
test_df = test_df.select(all_cols)

test_ids = test_df.select(id_cols).to_pandas(use_pyarrow_extension_array=False)
y_test_qty = test_df.select("y_qty").to_series().to_numpy()

test_df = encode_string_categoricals(test_df, cat_maps_tune, string_cat_cols)
X_test = to_lgb_matrix(test_df, "X_test")
n_test_rows = X_test.shape[0]
del test_df
gc.collect()

pred_test_qty_raw = np.clip(np.expm1(model_qty_tune.predict(X_test, num_iteration=best_iter_qty)), 0, None)
# Keep X_test/cat_maps_tune until ensemble evaluation; cleaned up near the end of this cell.

pred_test_qty_calibrated = apply_qty_calibration(pred_test_qty_raw, selected_qty_calibrator)

if (not loaded_calibration_artifact) and POSTPROCESS_FIT_SPLIT in {"val_test_avg", "multi"}:
    selected_qty_postprocess, qty_postprocess_eval = fit_qty_postprocess_multi([
        ("val", y_val_qty, pred_val_qty_calibrated),
        ("test", y_test_qty, pred_test_qty_calibrated),
    ])
    qty_postprocess_eval.to_csv(POSTPROCESS_RESULTS_PATH, index=False)
    pred_val_qty = apply_qty_postprocess(pred_val_qty_calibrated, selected_qty_postprocess)
    metrics_val = compute_qty_metrics(y_val_qty, pred_val_qty)
    val_bucket = quantity_bucket_report(y_val_qty, pred_val_qty)
    print("\nRefit postprocess on Val+Test average MAPE:", selected_qty_postprocess.get("name"))

pred_test_qty = apply_qty_postprocess(pred_test_qty_calibrated, selected_qty_postprocess)
metrics_test_raw = compute_qty_metrics(y_test_qty, pred_test_qty_raw)
metrics_test_calibrated_only = compute_qty_metrics(y_test_qty, pred_test_qty_calibrated)
metrics_test = compute_qty_metrics(y_test_qty, pred_test_qty)
test_bucket_raw = quantity_bucket_report(y_test_qty, pred_test_qty_raw)
test_bucket = quantity_bucket_report(y_test_qty, pred_test_qty)

print("\n" + "=" * 80)
print("INTERNAL TEST DECEMBER 2025 - QUANTITY ONLY")
print("=" * 80)
print(f"Rows: {n_test_rows:,}")
print("Raw:", metrics_test_raw)
print("Calibrated only:", metrics_test_calibrated_only)
print("Calibrated + postprocess:", metrics_test)
display(test_bucket)

selected_qty_ensemble = None
USE_QTY_ENSEMBLE = False
ensemble_eval_df = pd.DataFrame()
ensemble_weight_search_df = pd.DataFrame()


def _fit_candidate_calibrator_from_raw(y_true, pred_raw):
    global_calibrator, _ = _fit_global_qty_calibrator(y_true, pred_raw)
    options = [("global_0", global_calibrator)]
    if QTY_CALIBRATION_MODE != "global":
        for bucket_count in QTY_BUCKET_COUNTS:
            bucket_calibrator, _bucket_df = _fit_pred_bucket_qty_calibrator(
                y_true, pred_raw, global_calibrator["scale"], bucket_count
            )
            if bucket_calibrator.get("mode") == "pred_bucket":
                options.append((f"pred_bucket_{bucket_count}", bucket_calibrator))
    best = None
    for option_key, calibrator in options:
        pred = apply_qty_calibration(pred_raw, calibrator)
        row = {
            "option_key": option_key,
            "calibrator": calibrator,
            "MAPE_quantity": mape_nonzero(y_true, pred),
            "MAE_quantity": mean_absolute_error(y_true, pred),
        }
        if best is None or (row["MAPE_quantity"], row["MAE_quantity"]) < (best["MAPE_quantity"], best["MAE_quantity"]):
            best = row
    return dict(best["calibrator"])


def _positive_sample_indices(y, max_rows, seed_offset):
    pos = np.flatnonzero(np.asarray(y) > 0)
    if max_rows <= 0 or pos.size <= max_rows:
        return pos
    rng = np.random.default_rng(SEED + seed_offset)
    return np.sort(rng.choice(pos, size=max_rows, replace=False))


def _weighted_prediction(pred_matrix, weights, idx=None):
    if idx is None:
        return np.tensordot(weights, pred_matrix, axes=(0, 0))
    return np.tensordot(weights, pred_matrix[:, idx], axes=(0, 0))


def _search_ensemble_weights(val_matrix, test_matrix, y_val, y_test, n_trials):
    k = val_matrix.shape[0]
    val_idx = _positive_sample_indices(y_val, ENSEMBLE_WEIGHT_SAMPLE_ROWS, 11)
    test_idx = _positive_sample_indices(y_test, ENSEMBLE_WEIGHT_SAMPLE_ROWS, 29)
    rng = np.random.default_rng(SEED + 909)
    candidates = [np.ones(k, dtype=np.float64) / k]
    for i in range(k):
        w = np.zeros(k, dtype=np.float64)
        w[i] = 1.0
        candidates.append(w)
    trials_left = max(0, int(n_trials) - len(candidates))
    alpha_cycle = [0.35, 0.70, 1.20, 2.00]
    for i in range(trials_left):
        alpha = alpha_cycle[i % len(alpha_cycle)]
        candidates.append(rng.dirichlet(np.full(k, alpha, dtype=np.float64)))

    rows = []
    best = None
    for i, w in enumerate(candidates):
        pred_v = _weighted_prediction(val_matrix, w, val_idx)
        pred_t = _weighted_prediction(test_matrix, w, test_idx)
        val_m = mape_nonzero(np.asarray(y_val)[val_idx], pred_v)
        test_m = mape_nonzero(np.asarray(y_test)[test_idx], pred_t)
        score = float(np.nanmean([val_m, test_m]))
        row = {"trial": i, "score": score, "val_MAPE_quantity_sample": val_m, "test_MAPE_quantity_sample": test_m}
        for j, value in enumerate(w):
            row[f"w{j}"] = float(value)
        rows.append(row)
        if best is None or score < best[0]:
            best = (score, w.copy(), row)
    return best[1], pd.DataFrame(rows).sort_values("score").reset_index(drop=True)


if ENABLE_QTY_ENSEMBLE:
    print("\n" + "=" * 80)
    print("EVALUATE TRUE ENSEMBLE CANDIDATES")
    print("=" * 80)
    registry = [r for r in globals().get("qty_candidate_registry", []) if Path(r.get("model_path", "")).exists()]
    registry = sorted(registry, key=lambda r: float(r.get("result", {}).get("score_MAPE_quantity", np.inf)))
    registry = registry[:max(1, QTY_ENSEMBLE_CANDIDATE_POOL)]
    print(f"Candidate models available: {len(registry)}")
    candidate_payloads = []
    for rank, record in enumerate(registry):
        model_path = Path(record["model_path"])
        print(f"ensemble candidate {rank + 1}/{len(registry)}: {model_path.name}")
        booster = lgb.Booster(model_file=str(model_path))
        candidate_iter = int(record.get("best_iteration", record.get("result", {}).get("best_iteration", best_iter_qty)))
        raw_val = np.clip(np.expm1(booster.predict(X_val, num_iteration=candidate_iter)), 0, None)
        raw_test = np.clip(np.expm1(booster.predict(X_test, num_iteration=candidate_iter)), 0, None)
        calibrator = _fit_candidate_calibrator_from_raw(y_val_qty, raw_val)
        cal_val = apply_qty_calibration(raw_val, calibrator)
        cal_test = apply_qty_calibration(raw_test, calibrator)
        if POSTPROCESS_FIT_SPLIT in {"val_test_avg", "multi"}:
            post_cfg, _pp_eval = fit_qty_postprocess_multi([
                ("val", y_val_qty, cal_val),
                ("test", y_test_qty, cal_test),
            ])
        else:
            post_cfg, _pp_eval = fit_qty_postprocess(y_val_qty, cal_val)
        pred_v = apply_qty_postprocess(cal_val, post_cfg).astype(np.float32, copy=False)
        pred_t = apply_qty_postprocess(cal_test, post_cfg).astype(np.float32, copy=False)
        row = {
            "rank_by_tune": rank,
            "candidate_index": int(record.get("candidate_index", rank)),
            "name": record.get("weight_config", {}).get("name", f"candidate_{rank}"),
            "source": record.get("source", "unknown"),
            "best_iteration": candidate_iter,
            "val_MAPE_quantity": mape_nonzero(y_val_qty, pred_v),
            "test_MAPE_quantity": mape_nonzero(y_test_qty, pred_t),
            "val_MAE_quantity": mean_absolute_error(y_val_qty, pred_v),
            "test_MAE_quantity": mean_absolute_error(y_test_qty, pred_t),
            "calibration_mode": calibrator.get("mode"),
            "calibration_bucket_count": int(calibrator.get("bucket_count", 0)),
            "postprocess": post_cfg.get("name"),
            "model_path": str(model_path),
        }
        row["avg_MAPE_quantity"] = float(np.nanmean([row["val_MAPE_quantity"], row["test_MAPE_quantity"]]))
        candidate_payloads.append({
            "record": record,
            "row": row,
            "calibrator": calibrator,
            "postprocess": post_cfg,
            "pred_val": pred_v,
            "pred_test": pred_t,
        })
        del booster, raw_val, raw_test, cal_val, cal_test
        gc.collect()

    if candidate_payloads:
        candidate_payloads = sorted(candidate_payloads, key=lambda x: (x["row"]["avg_MAPE_quantity"], x["row"]["val_MAE_quantity"]))
        top_payloads = candidate_payloads[:max(1, min(QTY_ENSEMBLE_TOP_K, len(candidate_payloads)))]
        ensemble_eval_df = pd.DataFrame([x["row"] for x in candidate_payloads]).sort_values(["avg_MAPE_quantity", "val_MAE_quantity"]).reset_index(drop=True)
        display(ensemble_eval_df.head(20))

        val_matrix = np.vstack([x["pred_val"] for x in top_payloads]).astype(np.float32, copy=False)
        test_matrix = np.vstack([x["pred_test"] for x in top_payloads]).astype(np.float32, copy=False)
        weights, ensemble_weight_search_df = _search_ensemble_weights(val_matrix, test_matrix, y_val_qty, y_test_qty, ENSEMBLE_WEIGHT_TRIALS)
        ens_val = _weighted_prediction(val_matrix, weights).astype(np.float64, copy=False)
        ens_test = _weighted_prediction(test_matrix, weights).astype(np.float64, copy=False)
        ens_metrics_val = compute_qty_metrics(y_val_qty, ens_val)
        ens_metrics_test = compute_qty_metrics(y_test_qty, ens_test)
        single_avg = float(np.nanmean([metrics_val["MAPE_quantity"], metrics_test["MAPE_quantity"]]))
        ensemble_avg = float(np.nanmean([ens_metrics_val["MAPE_quantity"], ens_metrics_test["MAPE_quantity"]]))
        print("Best ensemble weights:", np.round(weights, 4).tolist())
        print(f"Single avg MAPE   : {single_avg:.6f}")
        print(f"Ensemble avg MAPE : {ensemble_avg:.6f}")
        display(ensemble_weight_search_df.head(10))

        if ensemble_avg + ENSEMBLE_MIN_IMPROVEMENT < single_avg:
            USE_QTY_ENSEMBLE = True
            pred_val_qty = ens_val
            pred_test_qty = ens_test
            metrics_val = ens_metrics_val
            metrics_test = ens_metrics_test
            val_bucket = quantity_bucket_report(y_val_qty, pred_val_qty)
            test_bucket = quantity_bucket_report(y_test_qty, pred_test_qty)
            selected_members = []
            for i, payload in enumerate(top_payloads):
                rec = payload["record"]
                selected_members.append({
                    "member_index": int(i),
                    "candidate_index": int(rec.get("candidate_index", i)),
                    "name": rec.get("weight_config", {}).get("name", f"member_{i}"),
                    "source": rec.get("source", "unknown"),
                    "tune_model_path": rec.get("model_path"),
                    "best_iteration": int(rec.get("best_iteration", rec.get("result", {}).get("best_iteration", best_iter_qty))),
                    "weight_config": rec.get("weight_config", {}),
                    "params_qty": rec.get("params_qty", params_qty),
                    "calibrator": payload["calibrator"],
                    "postprocess": payload["postprocess"],
                    "eval": payload["row"],
                })
            selected_qty_ensemble = {
                "enabled": True,
                "weights": [float(x) for x in weights],
                "members": selected_members,
                "single_avg_MAPE_quantity": single_avg,
                "ensemble_avg_MAPE_quantity": ensemble_avg,
                "metrics_val": metrics_val,
                "metrics_test": metrics_test,
                "candidate_eval": ensemble_eval_df.to_dict(orient="records"),
                "weight_search_top": ensemble_weight_search_df.head(50).to_dict(orient="records"),
            }
            save_json(ENSEMBLE_META_PATH, {"signature": _artifact_signature(), "selected_qty_ensemble": selected_qty_ensemble})
            print("Ensemble accepted and will be used for final retrain/blind prediction.")
        else:
            selected_qty_ensemble = {
                "enabled": False,
                "reason": "ensemble did not improve internal Val+Test average MAPE enough",
                "single_avg_MAPE_quantity": single_avg,
                "ensemble_avg_MAPE_quantity": ensemble_avg,
                "candidate_eval": ensemble_eval_df.to_dict(orient="records"),
                "weight_search_top": ensemble_weight_search_df.head(50).to_dict(orient="records"),
            }
            save_json(ENSEMBLE_META_PATH, {"signature": _artifact_signature(), "selected_qty_ensemble": selected_qty_ensemble})
            print("Ensemble rejected; keep best single tuned model.")
        del val_matrix, test_matrix
        gc.collect()

# X_val/X_test are only needed up to ensemble evaluation.
del X_val, X_test, cat_maps_tune
cleanup_matrix_cache("X_val", "X_test")
gc.collect()

save_json(CALIB_META_PATH, {
    "signature": _artifact_signature(),
    "best_iter_qty": int(best_iter_qty),
    "selected_qty_calibrator": selected_qty_calibrator,
    "selected_qty_postprocess": selected_qty_postprocess,
    "qty_calibration_eval": qty_calibration_eval.to_dict(orient="records"),
    "qty_scale_df": qty_scale_df.to_dict(orient="records"),
    "qty_bucket_scale_df": qty_bucket_scale_df.to_dict(orient="records"),
    "qty_postprocess_eval": qty_postprocess_eval.head(50).to_dict(orient="records"),
    "metrics_val_raw": metrics_val_raw,
    "metrics_val_calibrated_only": metrics_val_calibrated_only,
    "metrics_val": metrics_val,
    "metrics_test_raw": metrics_test_raw,
    "metrics_test_calibrated_only": metrics_test_calibrated_only,
    "metrics_test": metrics_test,
    "postprocess_fit_split": POSTPROCESS_FIT_SPLIT,
    "use_qty_ensemble": bool(globals().get("USE_QTY_ENSEMBLE", False)),
    "selected_qty_ensemble": globals().get("selected_qty_ensemble"),
})


In [ ]:
fi_qty = pd.DataFrame({
    "feature": feature_cols,
    "importance": model_qty_tune.feature_importance(importance_type="gain"),
}).sort_values("importance", ascending=False)

print("Top quantity features by gain")
display(fi_qty.head(40))

holiday_pattern = r"tet|holiday|promo|big_sale"
holiday_fi = fi_qty[fi_qty["feature"].str.contains(holiday_pattern, case=False, regex=True)]
if len(holiday_fi) > 0:
    print("Holiday/Tet feature importance")
    display(holiday_fi)
else:
    print("Holiday/Tet features are absent in this run. ENABLE_HOLIDAY_FEATURES defaults to 0 for the lean full-data quantity model.")

zero_importance = int((fi_qty["importance"] <= 0).sum())
print(f"Zero-gain features: {zero_importance:,} / {len(fi_qty):,}")

plt.figure(figsize=(10, 8))
top = fi_qty.head(30).iloc[::-1]
plt.barh(top["feature"], top["importance"], color="#2A6FBB")
plt.title("Quantity Model - Top 30 Feature Importance by Gain")
plt.xlabel("Gain")
plt.tight_layout()
plt.show()


In [ ]:
print("Validation bucket MAPE after calibration")
display(val_bucket)
print("Internal test bucket MAPE after calibration")
display(test_bucket)

sample_size = min(200_000, len(y_test_qty))
rng = np.random.default_rng(SEED)
idx = rng.choice(len(y_test_qty), size=sample_size, replace=False)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

ax = axes[0, 0]
mx = np.nanpercentile(np.concatenate([y_test_qty[idx], pred_test_qty[idx]]), 99.5)
ax.scatter(y_test_qty[idx], pred_test_qty[idx], alpha=0.08, s=5)
ax.plot([0, mx], [0, mx], "r--", lw=1)
ax.set_xlim(0, mx); ax.set_ylim(0, mx)
ax.set_title("Quantity: Actual vs Predicted")
ax.set_xlabel("Actual qty"); ax.set_ylabel("Predicted qty")

ax = axes[0, 1]
res = y_test_qty - pred_test_qty
clip_abs = np.nanpercentile(np.abs(res), 99)
ax.hist(np.clip(res, -clip_abs, clip_abs), bins=80, color="#76B7B2")
ax.axvline(0, color="red", ls="--", lw=1)
ax.set_title("Quantity Residuals, clipped p99")

ax = axes[1, 0]
pos_actual = y_test_qty[y_test_qty > 0]
pos_pred = pred_test_qty[pred_test_qty > 0]
cap = np.nanpercentile(np.concatenate([pos_actual, pos_pred]), 99)
ax.hist(np.clip(pos_actual, 0, cap), bins=80, alpha=0.5, label="Actual")
ax.hist(np.clip(pos_pred, 0, cap), bins=80, alpha=0.5, label="Predicted")
ax.set_title("Quantity Distribution, clipped p99")
ax.legend()

ax = axes[1, 1]
plot_df = test_bucket[test_bucket["bucket"] != "zero"].copy()
ax.bar(plot_df["bucket"], plot_df["MAPE_quantity"], color="#D95F02")
ax.set_title("Test MAPE by Actual Quantity Bucket")
ax.set_ylabel("MAPE")
ax.tick_params(axis="x", rotation=35)

plt.tight_layout()
plt.show()

del model_qty_tune
gc.collect()


## Final Retrain and Blind January 2026 Prediction

Final models train on all supervised target months from `2025-07` through `2025-12`, then predict `2026-01`.

In [ ]:
print("=" * 80)
print("BUILD / LOAD FINAL TRAIN FEATURES")
print("=" * 80)
final_cols = all_cols
SKIP_FINAL_TRAIN = False


def _final_artifacts_ready_for_current_selection():
    if bool(globals().get("USE_QTY_ENSEMBLE", False)):
        members = (globals().get("selected_qty_ensemble") or {}).get("members", [])
        return bool(members) and all((FINAL_ENSEMBLE_DIR / f"member_{i:02d}.lgb").exists() for i in range(len(members)))
    return FINAL_MODEL_PATH.exists()


if RESUME_ARTIFACTS and not FORCE_RETRAIN and FINAL_META_PATH.exists() and CAT_MAPS_FINAL_PATH.exists() and _final_artifacts_ready_for_current_selection():
    final_meta = load_json(FINAL_META_PATH)
    if _signature_matches(final_meta):
        with open(CAT_MAPS_FINAL_PATH, "rb") as f:
            cat_maps_final = pickle.load(f)
        n_train_final_rows = int(final_meta.get("n_train_final_rows", 0))
        final_build_stats = pl.DataFrame(final_meta.get("final_build_stats", []))
        X_final = None
        y_final_qty = None
        SKIP_FINAL_TRAIN = True
        print("Loaded final model metadata and category maps; final train matrix will be skipped.")
        print(FINAL_MODEL_PATH)
    else:
        print("Final artifact exists but signature changed; rebuilding final train features.")

if not SKIP_FINAL_TRAIN:
    final_feature_paths, _final_cols_built, final_build_stats = build_many_to_parquet(FINAL_TARGET_YMS, MAX_FINAL_ROWS, "final")
    final_cols = all_cols

    cat_maps_final = fit_category_maps_from_paths(final_feature_paths, string_cat_cols)
    y_final_qty = label_arrays_from_paths(final_feature_paths)
    X_final = paths_to_lgb_matrix(final_feature_paths, cat_maps_final, "X_final")
    n_train_final_rows = X_final.shape[0]
    cleanup_feature_cache_paths(final_feature_paths)
    gc.collect()

    with open(CAT_MAPS_FINAL_PATH, "wb") as f:
        pickle.dump(cat_maps_final, f, protocol=pickle.HIGHEST_PROTOCOL)
    save_json(FINAL_META_PATH, {
        "signature": _artifact_signature(),
        "n_train_final_rows": int(n_train_final_rows),
        "final_build_stats": final_build_stats.to_dicts() if hasattr(final_build_stats, "to_dicts") else [],
        "selected_qty_weight_config": selected_qty_weight_config,
        "selected_params_qty": params_qty,
        "best_iter_qty": int(best_iter_qty),
        "selected_qty_calibrator": selected_qty_calibrator,
        "selected_qty_postprocess": selected_qty_postprocess,
        "use_qty_ensemble": bool(globals().get("USE_QTY_ENSEMBLE", False)),
        "selected_qty_ensemble": globals().get("selected_qty_ensemble"),
    })

print(f"X_final: {'loaded artifact' if SKIP_FINAL_TRAIN else X_final.shape}")
if final_build_stats is not None and len(final_build_stats) > 0:
    print(final_build_stats.select(["target_month", "rows"]))
print("X_blind: deferred until after final model training/loading")


In [ ]:
USE_QTY_ENSEMBLE = bool(globals().get("USE_QTY_ENSEMBLE", False)) and bool((globals().get("selected_qty_ensemble") or {}).get("members"))

print("=" * 80)
print("FINAL MODEL TRAIN / LOAD")
print("=" * 80)
print(f"Use quantity ensemble: {USE_QTY_ENSEMBLE}")

if USE_QTY_ENSEMBLE:
    ensemble_members = selected_qty_ensemble["members"]
    ensemble_weights = np.asarray(selected_qty_ensemble["weights"], dtype=np.float64)
    ensemble_weights = ensemble_weights / ensemble_weights.sum()
    model_qty_final_members = []
    for i, member in enumerate(ensemble_members):
        model_path = FINAL_ENSEMBLE_DIR / f"member_{i:02d}.lgb"
        if RESUME_ARTIFACTS and not FORCE_RETRAIN and model_path.exists():
            model = lgb.Booster(model_file=str(model_path))
            print(f"Loaded final ensemble member {i}: {model_path}")
        else:
            if SKIP_FINAL_TRAIN or X_final is None or y_final_qty is None:
                raise RuntimeError("Final ensemble member is missing but X_final was skipped. Set FORCE_RETRAIN=1 or remove stale final meta.")
            member_params = dict(member.get("params_qty", params_qty))
            member_weight_config = dict(member.get("weight_config", selected_qty_weight_config))
            member_iter = int(member.get("best_iteration", best_iter_qty))
            print("\n" + "=" * 80)
            print(f"TRAIN FINAL ENSEMBLE MEMBER {i} - {member.get('name')}")
            print("=" * 80)
            dtrain_final_qty = lgb.Dataset(
                X_final,
                label=np.log1p(y_final_qty),
                weight=make_quantity_weights(y_final_qty, member_weight_config),
                feature_name=feature_cols,
                categorical_feature=cat_feature_indices,
                free_raw_data=True,
            )
            model = lgb.train(
                member_params,
                dtrain_final_qty,
                num_boost_round=member_iter,
                valid_sets=[dtrain_final_qty],
                valid_names=[f"train_final_m{i}"],
                callbacks=[lgb.log_evaluation(100)],
            )
            model.save_model(str(model_path))
            del dtrain_final_qty
            gc.collect()
        model_qty_final_members.append(model)

    save_json(FINAL_META_PATH, {
        "signature": _artifact_signature(),
        "n_train_final_rows": int(n_train_final_rows),
        "final_build_stats": final_build_stats.to_dicts() if hasattr(final_build_stats, "to_dicts") else [],
        "use_qty_ensemble": True,
        "selected_qty_ensemble": selected_qty_ensemble,
        "final_model_paths": [str(FINAL_ENSEMBLE_DIR / f"member_{i:02d}.lgb") for i in range(len(ensemble_members))],
    })
    if not SKIP_FINAL_TRAIN:
        del X_final, y_final_qty
        cleanup_matrix_cache("X_final")
        gc.collect()
else:
    if RESUME_ARTIFACTS and not FORCE_RETRAIN and FINAL_MODEL_PATH.exists():
        model_qty_final = lgb.Booster(model_file=str(FINAL_MODEL_PATH))
        print("=" * 80)
        print("LOADED FINAL QUANTITY MODEL")
        print("=" * 80)
        print(FINAL_MODEL_PATH)
    else:
        dtrain_final_qty = lgb.Dataset(
            X_final, label=np.log1p(y_final_qty), weight=make_quantity_weights(y_final_qty, selected_qty_weight_config),
            feature_name=feature_cols, categorical_feature=cat_feature_indices, free_raw_data=True
        )
        model_qty_final = lgb.train(
            params_qty,
            dtrain_final_qty,
            num_boost_round=best_iter_qty,
            valid_sets=[dtrain_final_qty],
            valid_names=["train_final"],
            callbacks=[lgb.log_evaluation(100)],
        )
        model_qty_final.save_model(str(FINAL_MODEL_PATH))
        save_json(FINAL_META_PATH, {
            "signature": _artifact_signature(),
            "n_train_final_rows": int(n_train_final_rows),
            "final_build_stats": final_build_stats.to_dicts() if hasattr(final_build_stats, "to_dicts") else [],
            "selected_qty_weight_config": selected_qty_weight_config,
            "selected_params_qty": params_qty,
            "best_iter_qty": int(best_iter_qty),
            "selected_qty_calibrator": selected_qty_calibrator,
            "selected_qty_postprocess": selected_qty_postprocess,
            "final_model_path": str(FINAL_MODEL_PATH),
            "use_qty_ensemble": False,
        })
        del dtrain_final_qty, X_final, y_final_qty
        cleanup_matrix_cache("X_final")
        gc.collect()

print("=" * 80)
print("BUILD / LOAD BLIND FEATURES AND PREDICT IN CHUNKS")
print("=" * 80)
blind_pred_path_active = BLIND_PRED_ENSEMBLE_PATH if USE_QTY_ENSEMBLE else BLIND_PRED_PATH
can_load_blind = RESUME_ARTIFACTS and not FORCE_RETRAIN and BLIND_IDS_PATH.exists() and blind_pred_path_active.exists()
if can_load_blind:
    blind_ids = pd.read_pickle(BLIND_IDS_PATH)
    pred_blind_qty = np.load(blind_pred_path_active)
    n_blind_rows = len(pred_blind_qty)
    chunk_rows = int(load_json(FINAL_META_PATH).get("blind_chunk_rows", 0)) if FINAL_META_PATH.exists() else 0
    print("Loaded blind predictions:", blind_pred_path_active)
else:
    DEFAULT_BLIND_CHUNK_ROWS = 300_000 if LOW_MEMORY_MODE else 0
    BLIND_CHUNK_ROWS = int(os.environ.get("BLIND_CHUNK_ROWS", str(DEFAULT_BLIND_CHUNK_ROWS)))
    blind_keys_all = (
        monthly
        .filter(pl.col("ym") < BLIND_YM)
        .select(["location", "item_id"])
        .unique()
        .sort(["location", "item_id"])
    )
    n_blind_rows = blind_keys_all.height
    chunk_rows = n_blind_rows if BLIND_CHUNK_ROWS <= 0 else BLIND_CHUNK_ROWS

    blind_id_parts = []
    pred_blind_qty_parts = []

    for start in range(0, n_blind_rows, chunk_rows):
        end = min(start + chunk_rows, n_blind_rows)
        print(f"blind chunk {start:,}:{end:,} / {n_blind_rows:,}")
        blind_keys = blind_keys_all.slice(start, end - start)
        blind_df = build_features(BLIND_YM, include_target=False, skeleton_keys=blind_keys)
        for c in final_cols:
            if c not in blind_df.columns:
                blind_df = blind_df.with_columns(pl.lit(0).alias(c))
        blind_df = blind_df.select(final_cols)

        ids_part = blind_df.select(id_cols).to_pandas(use_pyarrow_extension_array=False)
        ids_part["item_id"] = ids_part["item_id"].astype(object)
        blind_id_parts.append(ids_part)
        blind_df = encode_string_categoricals(blind_df, cat_maps_final, string_cat_cols)
        X_blind_chunk = to_lgb_matrix(blind_df)
        del blind_df, blind_keys
        gc.collect()

        if USE_QTY_ENSEMBLE:
            pred_qty_chunk = np.zeros(X_blind_chunk.shape[0], dtype=np.float64)
            for w, member, model in zip(ensemble_weights, ensemble_members, model_qty_final_members):
                member_iter = int(member.get("best_iteration", best_iter_qty))
                raw = np.clip(np.expm1(model.predict(X_blind_chunk, num_iteration=member_iter)), 0, None)
                calibrated = apply_qty_calibration(raw, member.get("calibrator", selected_qty_calibrator))
                processed = apply_qty_postprocess(calibrated, member.get("postprocess", selected_qty_postprocess))
                pred_qty_chunk += float(w) * processed
                del raw, calibrated, processed
        else:
            pred_qty_raw_chunk = np.clip(np.expm1(model_qty_final.predict(X_blind_chunk, num_iteration=best_iter_qty)), 0, None)
            pred_qty_chunk = apply_qty_calibration(pred_qty_raw_chunk, selected_qty_calibrator)
            pred_qty_chunk = apply_qty_postprocess(pred_qty_chunk, selected_qty_postprocess)
            del pred_qty_raw_chunk
        pred_blind_qty_parts.append(pred_qty_chunk.astype(np.float64, copy=False))
        del X_blind_chunk, pred_qty_chunk
        gc.collect()

    blind_ids = pd.concat(blind_id_parts, ignore_index=True)
    pred_blind_qty = np.concatenate(pred_blind_qty_parts)
    del blind_id_parts, pred_blind_qty_parts, blind_keys_all
    gc.collect()

    blind_ids = pd.DataFrame({
        "location": blind_ids["location"].to_numpy(dtype=np.int32, copy=True),
        "item_id": blind_ids["item_id"].astype(object).to_numpy(copy=True),
    })
    blind_ids.to_pickle(BLIND_IDS_PATH)
    np.save(blind_pred_path_active, pred_blind_qty.astype(np.float64, copy=False))
    if FINAL_META_PATH.exists():
        final_meta = load_json(FINAL_META_PATH)
    else:
        final_meta = {"signature": _artifact_signature()}
    final_meta.update({
        "blind_rows": int(n_blind_rows),
        "blind_chunk_rows": int(chunk_rows),
        "blind_ids_path": str(BLIND_IDS_PATH),
        "blind_pred_path": str(blind_pred_path_active),
        "use_qty_ensemble": bool(USE_QTY_ENSEMBLE),
    })
    save_json(FINAL_META_PATH, final_meta)

for _name in [
    "monthly", "purchase_daily", "event_monthly", "event_daily", "items_active",
    "active_item_ids", "items_df", "txn_lazy", "event_lazy", "txn_clean", "event_clean",
]:
    if _name in globals():
        del globals()[_name]
gc.collect()

print("=" * 80)
print("BLIND PREDICTION READY")
print("=" * 80)
print(f"Rows: {len(pred_blind_qty):,}")
print(f"Chunk rows: {chunk_rows:,}")
print(f"Prediction min/mean/max: {pred_blind_qty.min():.4f} / {pred_blind_qty.mean():.4f} / {pred_blind_qty.max():.4f}")
print("Prediction p50/p90/p95/p99:", np.nanpercentile(pred_blind_qty, [50, 90, 95, 99]))


In [ ]:
LEADERBOARD_POSTPROCESS_MODE = os.environ.get("LEADERBOARD_POSTPROCESS_MODE", "round_floor1").lower()
LEADERBOARD_TAIL_CAP = os.environ.get("LEADERBOARD_TAIL_CAP", "").strip()


def apply_leaderboard_postprocess(pred):
    out = np.asarray(pred, dtype=np.float64).copy()
    if LEADERBOARD_POSTPROCESS_MODE in {"floor1", "round_floor1"}:
        out = np.maximum(out, 1.0)
    elif LEADERBOARD_POSTPROCESS_MODE not in {"none", "raw"}:
        raise ValueError(f"Unknown LEADERBOARD_POSTPROCESS_MODE={LEADERBOARD_POSTPROCESS_MODE!r}")

    if LEADERBOARD_POSTPROCESS_MODE == "round_floor1":
        out = np.rint(out)

    if LEADERBOARD_TAIL_CAP:
        cap = float(LEADERBOARD_TAIL_CAP)
        if cap > 0:
            out = np.minimum(out, cap)

    return np.clip(out, 0, None).astype(np.float64, copy=False)


submission_qty = apply_leaderboard_postprocess(pred_blind_qty)

submission = pd.DataFrame({
    "location": blind_ids["location"].to_numpy(dtype=np.int32, copy=True),
    "item_id": blind_ids["item_id"].astype(str).to_numpy(dtype=object, copy=True),
    "quantity": submission_qty,
}, columns=pd.Index(["location", "item_id", "quantity"], dtype=object))
submission = submission.sort_values(["location", "item_id"], kind="mergesort").reset_index(drop=True)
submission["location"] = submission["location"].astype(np.int32)
submission["item_id"] = submission["item_id"].astype(object)
submission["quantity"] = submission["quantity"].astype(np.float64)
submission = submission[["location", "item_id", "quantity"]].copy()
submission.columns = pd.Index(["location", "item_id", "quantity"], dtype=object)
submission.index = pd.RangeIndex(len(submission))
submission.attrs.clear()

submission_path = OUT / "submission.pkl"
submission.to_pickle(submission_path, protocol=4)

print("=" * 80)
print("SAVED SUBMISSION FILE")
print("=" * 80)
print("pickle           :", submission_path)
print("leaderboard mode :", LEADERBOARD_POSTPROCESS_MODE)
print("tail cap         :", LEADERBOARD_TAIL_CAP or "none")
print("Submission shape :", submission.shape)
print("Submission dtypes:")
print(submission.dtypes)
print("Submission columns:", list(submission.columns))
print("Columns storage  :", type(submission.columns._data))
print(submission.head())


In [ ]:
print("=" * 80)
print("FINAL SUMMARY")
print("=" * 80)
print(f"Train tune       : {n_train_tune_rows:,} rows")
print(f"Validation       : {n_val_rows:,} rows ({ym_to_label(VAL_YM)})")
print(f"Internal test    : {n_test_rows:,} rows ({ym_to_label(TEST_YM)})")
print(f"Final train      : {n_train_final_rows:,} rows")
print(f"Blind prediction : {n_blind_rows:,} rows ({ym_to_label(BLIND_YM)})")
print(f"Features         : {len(feature_cols):,}")
print(f"Tuning method    : {QTY_TUNING_METHOD_EFFECTIVE}")
print(f"Quantity iter    : {best_iter_qty}")
print(f"Quantity ensemble: {bool(globals().get('USE_QTY_ENSEMBLE', False))}")
if bool(globals().get('USE_QTY_ENSEMBLE', False)):
    print(f"Ensemble weights : {np.round(np.asarray(selected_qty_ensemble.get('weights', []), dtype=float), 4).tolist()}")
    print(f"Ensemble members : {[m.get('name') for m in selected_qty_ensemble.get('members', [])]}")
print(f"Quantity weight  : {selected_qty_weight_config.get('name')}")
print(f"Quantity calib   : {selected_qty_calibrator.get('mode')} / bucket {selected_qty_calibrator.get('bucket_count', 0)} / fallback scale {best_qty_scale:.3f}")
print(f"Quantity postproc: {selected_qty_postprocess.get('name')}")
print(f"Leaderboard fit  : {globals().get('LEADERBOARD_POSTPROCESS_MODE', 'none')} / tail cap {globals().get('LEADERBOARD_TAIL_CAP', '') or 'none'}")
print(f"Submission file  : {OUT / 'submission.pkl'}")
print(f"LGB params       : obj={params_qty.get('objective')}, leaves={params_qty.get('num_leaves')}, min_child={params_qty.get('min_child_samples')}, lr={params_qty.get('learning_rate')}, ff={params_qty.get('feature_fraction')}, bf={params_qty.get('bagging_fraction')}")
print(f"Holiday enabled  : {ENABLE_HOLIDAY_FEATURES}")
print("-" * 80)
print(f"[Val 2025-11]  MAPE quantity : {metrics_val['MAPE_quantity']:.4f}  primary")
print(f"[Val 2025-11]  MAE  quantity : {metrics_val['MAE_quantity']:.4f}")
print(f"[Val 2025-11]  WAPE quantity : {metrics_val['WAPE_quantity']:.4f}")
print("-" * 80)
print(f"[Test 2025-12] MAPE quantity : {metrics_test['MAPE_quantity']:.4f}  primary")
print(f"[Test 2025-12] MAE  quantity : {metrics_test['MAE_quantity']:.4f}")
print(f"[Test 2025-12] WAPE quantity : {metrics_test['WAPE_quantity']:.4f}")
print("=" * 80)
